# Assignment 3: From CATE Estimation to Budget-Constrained Targeting

## Overview

In previous coding sessions, we built a full causal ML pipeline: meta-learners, causal trees and forests, and double machine learning. This assignment assembles those pieces into a single end-to-end decision pipeline on the **Starbucks Offers dataset**, using the production libraries **`econml`** and **`lightgbm`** with **hyperparameter tuning** at every stage.

**Outcome variable: incremental spending.** Rather than binary conversion, the outcome $Y_i$ is each customer's **total dollar spending** in the 14-day observation window. The CATE $\hat{\tau}(x) = E[Y(1) - Y(0) \mid X=x]$ is therefore measured in **dollars of incremental spending** — directly comparable to the coupon cost $c$ with no revenue multiplier needed. This makes the optimization economically grounded: a customer is worth targeting if and only if the expected incremental spending exceeds the coupon cost.

The pipeline has four stages:

| Stage | Task | Parts |
|-------|------|-------|
| **1. Train** | Fit and tune 13 CATE models on the training set | Parts 1–3 |
| **2. Evaluate** | Score all 13 on the held-out test set | Part 4 |
| **3. Select** | Pick one winner and justify | Part 4 |
| **4. Optimize** | Design the campaign with the winner's CATE estimates | Part 5 |

**Ground rules:**
- Parts 1–3: train and tune using only the training set. No peeking at test-set outcomes.
- Part 4: the only place where test-set outcomes are used for comparison.
- Part 5: uses **only** `cate_winner`. Do not substitute another model.

**What to submit:** This notebook with all cells run and all written answers filled in. Verify with Kernel → Restart & Run All before submitting.

---
## Setup

**Do not modify this cell.**

In [ ]:
!pip install econml

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
from scipy.optimize import linprog

# econml
from econml.grf import CausalForest
from econml.dml import LinearDML, NonParamDML, CausalForestDML

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 110,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size': 11,
})
print('Setup complete.')

---
## Data Loading and Preparation

**Do not modify this cell.**

**Quasi-experimental design (same as Lab 4).** Customers whose first offer arrived before the median timing are the **treated group** (spending measured in the 14 days after receipt). Customers whose first offer arrived after the median are the **control group** (spending measured in the 14 days before receipt).

**Selection problem.** Treated customers tend to be higher organic spenders — they visit Starbucks more frequently and are more likely to transact in any window regardless of the offer. A pre-treatment spending rate covariate is not feasible here: the treated group's first offer often arrives so early that they have no observable pre-treatment window, making the covariate structurally zero for most treated customers and asymmetric by construction. Instead, we rely on the available **profile-level covariates** — age, income, and gender — which are measured at registration and are genuinely pre-treatment and symmetric across arms. The quasi-random timing of offer arrival, conditional on these demographics, provides the identification.

**Variables:**
- `Y`: continuous — total dollar spending in the 14-day observation window (0 if no transactions)
- `D`: binary — 1 if treated (early offer recipient)
- `X`: age, income, gender dummies, missingness indicator

In [ ]:
import kagglehub
path = kagglehub.dataset_download("mexwell/starbucks-offers-advertisement-data")

profile    = pd.read_json(path + "/profile.json",    orient="records", lines=True)
transcript = pd.read_json(path + "/transcript.json", orient="records", lines=True)

# ── Parse transcript ──────────────────────────────────────────────────────────
trx = pd.concat([transcript.drop('value', axis=1),
                 transcript['value'].apply(pd.Series)], axis=1)
trx['offer_id'] = trx.get('offer id')
offers       = trx[trx['event'] == 'offer received'][['person', 'offer_id', 'time']]
transactions = trx[trx['event'] == 'transaction'][['person', 'time', 'amount']]

# ── First offer time and window parameters ────────────────────────────────────
first_offer         = offers.groupby('person')['time'].min().reset_index()
first_offer.columns = ['person', 'first_offer_time']
median_time         = first_offer['first_offer_time'].median()
window_end          = median_time + 14
early = first_offer[first_offer['first_offer_time'] <= median_time]
late  = first_offer[first_offer['first_offer_time'] >  median_time]

# ── Build spending outcome per arm ────────────────────────────────────────────
def build_arm(cohort, transactions, after, window_end):
    """
    Outcome Y = total dollar spending in the 14-day observation window.
    Treated arm: spending after first offer receipt.
    Control arm: spending before first offer receipt (organic baseline).
    Customers with no transactions get Y = 0.
    """
    tx   = transactions.merge(cohort, on='person')
    mask = ((tx['time'] > tx['first_offer_time']) if after
            else (tx['time'] < tx['first_offer_time'])) & (tx['time'] <= window_end)
    spend = tx[mask].groupby('person')['amount'].sum().reset_index()
    spend.columns = ['person', 'total_spend']
    out  = cohort.merge(profile, left_on='person', right_on='id')
    out  = out.merge(spend, on='person', how='left')
    out['total_spend'] = out['total_spend'].fillna(0.0)
    return out[['person', 'age', 'gender', 'income', 'total_spend']]

treated_raw = build_arm(early, transactions, after=True,  window_end=window_end)
control_raw = build_arm(late,  transactions, after=False, window_end=window_end)
treated_raw['treatment'] = 1
control_raw['treatment'] = 0

df = pd.concat([
    treated_raw.sample(50000, replace=True, random_state=RANDOM_STATE),
    control_raw.sample(50000, replace=True, random_state=RANDOM_STATE)
], ignore_index=True)

# ── Feature engineering ───────────────────────────────────────────────────────
df.loc[df['age'] == 118, 'age'] = np.nan
df['missing_demo'] = df['income'].isna().astype(int)
df['age']    = df['age'].fillna(df['age'].median())
df['income'] = df['income'].fillna(df['income'].median())
df['gender'] = df['gender'].fillna('U')
df = pd.get_dummies(df, columns=['gender'], prefix='gender', drop_first=False)
df = df.drop(columns=['gender_U'])   # gender_U as reference category
for c in [col for col in df.columns if col.startswith('gender_')]:
    df[c] = df[c].astype(int)

FEATURE_COLS = ['age', 'income'] + sorted(
    [c for c in df.columns if c.startswith('gender_')]) + ['missing_demo']

# ── Train / test split (70 / 30, stratified by treatment) ────────────────────
X_all = df[FEATURE_COLS + ['treatment']].astype(float)
y_all = df['total_spend']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.3, random_state=RANDOM_STATE,
    stratify=df.loc[X_all.index, 'treatment'])

D_tr  = X_tr['treatment'].values;   D_te  = X_te['treatment'].values
Xf_tr = X_tr[FEATURE_COLS].values;  Xf_te = X_te[FEATURE_COLS].values
Y_tr  = y_tr.values;                Y_te  = y_te.values

scaler = StandardScaler()
Xs_tr  = scaler.fit_transform(Xf_tr)
Xs_te  = scaler.transform(Xf_te)

print(f"Features : {FEATURE_COLS}")
print(f"Train    : {len(X_tr):,}  (treated {(D_tr==1).sum():,} / control {(D_tr==0).sum():,})")
print(f"Test     : {len(X_te):,}  (treated {(D_te==1).sum():,} / control {(D_te==0).sum():,})")
print(f"Y (spending) — train treated mean: ${Y_tr[D_tr==1].mean():.2f}  "
      f"control mean: ${Y_tr[D_tr==0].mean():.2f}")
print(f"Naive ATE (diff-in-means): ${Y_tr[D_tr==1].mean() - Y_tr[D_tr==0].mean():.2f}")


### Business Parameters and Shared Helpers

**Do not modify this cell.**

Each coupon costs **\$c = 2.00** to send. The outcome $Y_i$ is measured in **dollars of spending**, so the CATE $\hat{\tau}(x) = E[Y(1)-Y(0)\mid X=x]$ is already in dollars. A customer is worth targeting if and only if $\hat{\tau}_i > c$ — no revenue multiplier needed.

**Break-even CATE: \$2.00.** Any customer whose estimated incremental spending exceeds the coupon cost is profitable to target.

In [ ]:
COUPON_COST = 2.0
BREAK_EVEN  = COUPON_COST   # hat_tau_i > COUPON_COST to be worth targeting

# ── Shared naive hyperparameters ─────────────────────────────────────────────
NAIVE_LGB = dict(n_estimators=200, max_depth=4, num_leaves=31,
                 learning_rate=0.05, min_child_samples=20,
                 verbose=-1, random_state=RANDOM_STATE)
NAIVE_CF  = dict(n_estimators=1000, min_samples_leaf=10,
                 max_depth=None, honest=True, random_state=RANDOM_STATE)

# ── Tuning grid for LightGBM ──────────────────────────────────────────────────
LGB_GRID = {
    'num_leaves':        [15, 30, 60],
    'max_depth':         [3, 4, 6],
    'min_child_samples': [20, 50],
    'learning_rate':     [0.05, 0.1],
}

# ── AUUC (Qini) ───────────────────────────────────────────────────────────────
def compute_auuc(y_true, treatment, uplift_scores):
    """Returns (auuc, fractions, qini_values)."""
    order  = np.argsort(-uplift_scores)
    y_s, d_s = y_true[order], treatment[order]
    n, n_t, n_c = len(y_s), d_s.sum(), len(y_s) - d_s.sum()
    cum_t  = np.cumsum(y_s * d_s)
    cum_c  = np.cumsum(y_s * (1 - d_s))
    qini   = (cum_t / n_t - cum_c / n_c) * n
    fracs  = np.insert(np.arange(1, n + 1) / n, 0, 0)
    qini   = np.insert(qini, 0, 0)
    random = np.linspace(0, qini[-1], len(fracs))
    return float(np.trapz(qini - random, fracs)), fracs, qini

# ── Profit evaluation — two methods ──────────────────────────────────────────
def profit_model_based(cate_hat, target_mask):
    """
    Method 1: assume hat_tau_i is the true incremental spending.
    profit = sum_{i in T} (hat_tau_i - c)
    Circular: optimized and evaluated with the same estimates.
    Y is in dollars, so no revenue multiplier is needed.
    """
    return float(np.sum(cate_hat[target_mask] - COUPON_COST))

def profit_empirical(y_true, treatment, target_mask):
    """
    Method 2: difference-in-means of actual spending within the targeted group.

    The targeting rule is applied to ALL test customers (both arms), partitioning
    the test set into targeted T and untargeted U. Within T, D is quasi-random
    (targeting depends only on X, not D or Y), so:
        hat_tau_T = mean(Y | D=1, i in T) - mean(Y | D=0, i in T)
    is an unbiased estimate of E[Y(1)-Y(0) | X in T] in dollar terms.

    Required assumptions:
      1. Unconfoundedness: D ⊥ Y(0),Y(1) | X  (quasi-random treatment, supported
         by the quasi-random timing of offer arrival conditional on demographics)
      2. Pre-outcome targeting: hat_tau_i depends only on X_i (training-set model)
      3. Overlap within T: both arms present inside the targeted group
      4. SUTVA: no spillovers between customers

    Returns dict: n_targeted, n_treated_in_T, n_control_in_T,
                  tau_empirical ($), profit ($)
    """
    if target_mask.sum() == 0:
        return dict(n_targeted=0, n_treated_in_T=0, n_control_in_T=0,
                    tau_empirical=np.nan, profit=0.0)
    yt = y_true[target_mask]
    dt = treatment[target_mask]
    n_t = int(dt.sum())
    n_c = int((1 - dt).sum())
    if n_t == 0 or n_c == 0:
        return dict(n_targeted=int(target_mask.sum()), n_treated_in_T=n_t,
                    n_control_in_T=n_c, tau_empirical=np.nan,
                    profit=float(-COUPON_COST * target_mask.sum()))
    tau_emp = yt[dt == 1].mean() - yt[dt == 0].mean()   # dollars
    profit  = float((tau_emp - COUPON_COST) * target_mask.sum())
    return dict(n_targeted=int(target_mask.sum()), n_treated_in_T=n_t,
                n_control_in_T=n_c, tau_empirical=float(tau_emp),
                profit=profit)

print(f"Break-even CATE : ${BREAK_EVEN:.2f} (incremental spending must exceed coupon cost)")
print("Helpers ready   : compute_auuc(), profit_model_based(), profit_empirical()")

---
# STAGE 1 — TRAIN

Train and tune 13 CATE models on `(Xf_tr, Xs_tr, Y_tr, D_tr)`. **Do not use the test set.**

**Outcome is continuous spending ($Y \geq 0$).** This has two implications for model choices:
- **S-Learner and T-Learner** use `lgb.LGBMRegressor` (not classifier). Tune by **cross-validated RMSE** (`neg_root_mean_squared_error`).
- **DML nuisance `model_y`** uses `lgb.LGBMRegressor`, tuned by cross-validated **MSE** (`neg_mean_squared_error`). The treatment nuisance `model_t` remains a classifier (D is still binary), tuned by log-loss.
- **`CausalForest` and `CausalForestDML`** handle continuous outcomes natively.

Everything else — the tuning logic, the LP optimization, the profit evaluation — is the same as with binary outcomes, but now in dollar units throughout. Note that `FEATURE_COLS` contains only profile-level demographics (age, income, gender dummies, missingness indicator) — no pre-treatment spending covariate.

In [ ]:
# YOUR CODE HERE
# ── Part 1a: Linear Interaction Model ───────────────────────────────────────
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Build design matrix: [X, D, X*D]
Xf_d = np.column_stack([Xf_tr, D_tr])             # X plus treatment
inter = Xf_tr * D_tr[:, None]                    # interaction terms X * D
X_ols = np.column_stack([Xf_d, inter])
X_ols = sm.add_constant(X_ols)

# Feature names for the regression table
feat_names = FEATURE_COLS + ['treatment'] + [f'{f}:D' for f in FEATURE_COLS]
feat_names = ['const'] + feat_names

# Fit OLS with robust standard errors
model_ols = sm.OLS(Y_tr, X_ols).fit(cov_type='HC1')
print(model_ols.summary(xname=feat_names))

# Significant interaction terms at 5%
coef_df = pd.DataFrame({
    'coef': model_ols.params,
    'pvalue': model_ols.pvalues
}, index=feat_names)

sig_inter = coef_df[
    coef_df.index.str.endswith(':D') & (coef_df['pvalue'] < 0.05)
]

print("\nSignificant interaction terms (p < 0.05):")
print(sig_inter)

# Function to predict potential outcomes under D = 1 or D = 0
def predict_ols(X_feat, d_val):
    Xd = np.column_stack([X_feat, np.full(len(X_feat), d_val)])
    int_ = X_feat * d_val
    Xols = np.column_stack([np.ones(len(X_feat)), Xd, int_])
    return model_ols.predict(Xols)

# Training-set CATE
cate_tr_linear = predict_ols(Xf_tr, 1) - predict_ols(Xf_tr, 0)
print(f"\nTraining CATE — mean: ${cate_tr_linear.mean():.3f}  std: ${cate_tr_linear.std():.3f}")

# Test-set CATE predictions
cate_linear = predict_ols(Xf_te, 1) - predict_ols(Xf_te, 0)
print(f"Test CATE     — mean: ${cate_linear.mean():.3f}  std: ${cate_linear.std():.3f}")

# Check required output
print("\ncate_linear shape:", cate_linear.shape)

## Part 1: Linear Benchmark and Meta-Learners (25 points)

### 1a. Linear Interaction Model (5 points)

Fit an OLS regression of `Y` (total spending, continuous) on `[X, D, X*D]` using `statsmodels.OLS`. Predicted CATE: $\hat{\tau}_{\text{linear}}(x) = \hat{f}(x, D=1) - \hat{f}(x, D=0)$, in dollars.

**Report:** full coefficient table, which interaction terms are significant at 5%, mean and std of training-set CATE.

**Store** test-set predictions as `cate_linear` (numpy array, length $N_{\text{test}}$).

💻 **Code required**

### 1b. S-Learner: Naive and Tuned (10 points)

Train one `lgb.LGBMRegressor` on all training data with `treatment` as a feature. Predicted CATE:

$$\hat{\tau}_{\text{S}}(x) = \hat{E}[Y \mid x, D=1] - \hat{E}[Y \mid x, D=0]$$

**Tuning rationale.** Predicting spending $Y$ from $(X, D)$ is a regression problem. Tune by **cross-validated RMSE** (`scoring='neg_root_mean_squared_error'`) using `GridSearchCV` with `LGB_GRID`.

**Step 1 — Naive.** Fit `lgb.LGBMRegressor(**NAIVE_LGB)` on `[Xf_tr, D_tr]` stacked. Store test CATE as `cate_s_naive`.

**Step 2 — Tune.** `GridSearchCV` (5-fold, RMSE) on the same feature matrix. Print best params and best CV RMSE.

**Step 3 — Tuned.** Refit with best params. Store as `cate_s_tuned`.

**Step 4 — Compare.** Print mean and std of `cate_s_naive` and `cate_s_tuned`. Plot overlapping histograms of both CATE distributions, with vertical dashed lines at their respective means and a dotted line at the break-even threshold.

✍️ **Written answer** (2–3 sentences): Did tuning change the CATE distribution substantially? The CATE is a difference of two spending predictions — explain why lower RMSE on the outcome model does not guarantee better CATE ranking, and what feature interaction would need to be better captured for tuning to reliably improve CATE quality.

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# cate_s_naive  = np.array(...)    ← required
# cate_s_tuned  = np.array(...)    ← required

# ── Part 1b: S-Learner Naive and Tuned ───────────────────────────────────────

# S-learner uses treatment as one additional feature
X_s_tr = np.column_stack([Xf_tr, D_tr])

# Test features under D = 1 and D = 0
X_s_te1 = np.column_stack([Xf_te, np.ones(len(Xf_te))])
X_s_te0 = np.column_stack([Xf_te, np.zeros(len(Xf_te))])

# Step 1 — Naive S-Learner
s_naive = lgb.LGBMRegressor(**NAIVE_LGB)
s_naive.fit(X_s_tr, Y_tr)

cate_s_naive = s_naive.predict(X_s_te1) - s_naive.predict(X_s_te0)

print(f"S-Learner naive -- mean: ${cate_s_naive.mean():.3f}  std: ${cate_s_naive.std():.3f}")

# Step 2 — Tune by 5-fold CV RMSE
gs_s = GridSearchCV(
    estimator=lgb.LGBMRegressor(
        n_estimators=200,
        verbose=-1,
        random_state=RANDOM_STATE
    ),
    param_grid=LGB_GRID,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

gs_s.fit(X_s_tr, Y_tr)

print("\nS-Learner tuned best params:")
print(gs_s.best_params_)
print(f"Best CV RMSE: ${-gs_s.best_score_:.4f}")

# Step 3 — Tuned S-Learner
s_tuned = lgb.LGBMRegressor(
    n_estimators=200,
    verbose=-1,
    random_state=RANDOM_STATE,
    **gs_s.best_params_
)

s_tuned.fit(X_s_tr, Y_tr)

cate_s_tuned = s_tuned.predict(X_s_te1) - s_tuned.predict(X_s_te0)

print(f"\nS-Learner tuned -- mean: ${cate_s_tuned.mean():.3f}  std: ${cate_s_tuned.std():.3f}")

# Step 4 — Compare
summary_s = pd.DataFrame({
    "model": ["S-Learner naive", "S-Learner tuned"],
    "mean_CATE": [cate_s_naive.mean(), cate_s_tuned.mean()],
    "std_CATE": [cate_s_naive.std(), cate_s_tuned.std()],
    "share_above_break_even": [
        np.mean(cate_s_naive > BREAK_EVEN),
        np.mean(cate_s_tuned > BREAK_EVEN)
    ]
})

print("\nS-Learner CATE summary:")
display(summary_s)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(cate_s_naive, bins=50, alpha=0.6, label="Naive")
ax.hist(cate_s_tuned, bins=50, alpha=0.6, label="Tuned")

ax.axvline(cate_s_naive.mean(), linestyle="--", linewidth=1.5, label="Naive mean")
ax.axvline(cate_s_tuned.mean(), linestyle="--", linewidth=1.5, label="Tuned mean")
ax.axvline(BREAK_EVEN, linestyle=":", linewidth=2, label=f"Break-even = ${BREAK_EVEN:.2f}")

ax.set_title("S-Learner CATE Distribution")
ax.set_xlabel("Predicted CATE ($)")
ax.set_ylabel("Number of customers")
ax.legend()
plt.tight_layout()
plt.show()

assert isinstance(cate_s_naive, np.ndarray)
assert isinstance(cate_s_tuned, np.ndarray)
assert cate_s_naive.shape == (len(Xf_te),)
assert cate_s_tuned.shape == (len(Xf_te),)

print("\nRequired outputs created:")
print("cate_s_naive shape:", cate_s_naive.shape)
print("cate_s_tuned shape:", cate_s_tuned.shape)

*Your answer:*Tuning did not change the average S-Learner CATE very much, since the naive and tuned means are both around $0.52. However, it changed the distribution substantially by increasing the standard deviation from about $0.58 to about $1.50 and increasing the share of customers above the $2 break-even threshold from about 1.0% to 6.4%. A lower RMSE for predicting spending does not guarantee better CATE ranking, because CATE is the difference between two predicted spending outcomes under D=1 and D=0; tuning only reliably improves CATE quality if it better captures the interactions between treatment and customer features that drive treatment-effect heterogeneity.

### 1c. T-Learner: Naive and Tuned (10 points)

Train two separate `lgb.LGBMRegressor` models — one per arm — using only `FEATURE_COLS`. Predicted CATE: $\hat{\tau}_{\text{T}}(x) = \hat{\mu}_1(x) - \hat{\mu}_0(x)$ in dollars.

**Tuning rationale.** Each arm model is an independent regression. Tune them independently by **cross-validated RMSE** on their respective arm subsets.

**Step 1 — Naive.** Two `lgb.LGBMRegressor(**NAIVE_LGB)`, one per arm. Store as `cate_t_naive`.

**Step 2 — Tune each arm independently.** `GridSearchCV` (5-fold, RMSE) on `(Xf_tr[D_tr==1], Y_tr[D_tr==1])` and `(Xf_tr[D_tr==0], Y_tr[D_tr==0])` separately. Print best params for each arm.

**Step 3 — Tuned.** Refit each arm with its best params. Store as `cate_t_tuned`.

**Step 4 — Compare.** Print mean and std of both. Plot overlapping histograms of naive vs. tuned CATE distributions.

✍️ **Written answer** (2–3 sentences): Did the two arms converge to the same best hyperparameters? The treated arm's spending distribution may be heavier-tailed than the control arm's — if the best `num_leaves` differs between arms, what does this suggest about the complexity of each arm's conditional spending surface?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# cate_t_naive  = np.array(...)    ← required
# cate_t_tuned  = np.array(...)    ← required

# ── Part 1c: T-Learner Naive and Tuned ───────────────────────────────────────

from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

# Split training data by treatment arm
mask_t = (D_tr == 1)
mask_c = (D_tr == 0)

X_treat, y_treat = Xf_tr[mask_t], Y_tr[mask_t]
X_ctrl,  y_ctrl  = Xf_tr[mask_c], Y_tr[mask_c]

print("Treated arm size:", X_treat.shape[0])
print("Control arm size:", X_ctrl.shape[0])

# Step 1 — Naive T-Learner
treat_naive = lgb.LGBMRegressor(**NAIVE_LGB)
ctrl_naive  = lgb.LGBMRegressor(**NAIVE_LGB)

treat_naive.fit(X_treat, y_treat)
ctrl_naive.fit(X_ctrl, y_ctrl)

mu1_naive = treat_naive.predict(Xf_te)
mu0_naive = ctrl_naive.predict(Xf_te)

cate_t_naive = mu1_naive - mu0_naive

print(f"\nT-Learner naive -- mean: ${cate_t_naive.mean():.3f}  std: ${cate_t_naive.std():.3f}")

# Step 2 — Tune each arm independently by CV RMSE
gs_treat = GridSearchCV(
    estimator=lgb.LGBMRegressor(
        n_estimators=200,
        verbose=-1,
        random_state=RANDOM_STATE
    ),
    param_grid=LGB_GRID,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

gs_ctrl = GridSearchCV(
    estimator=lgb.LGBMRegressor(
        n_estimators=200,
        verbose=-1,
        random_state=RANDOM_STATE
    ),
    param_grid=LGB_GRID,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

gs_treat.fit(X_treat, y_treat)
gs_ctrl.fit(X_ctrl, y_ctrl)

print("\nTreated arm best params:")
print(gs_treat.best_params_)
print(f"Treated arm best CV RMSE: ${-gs_treat.best_score_:.4f}")

print("\nControl arm best params:")
print(gs_ctrl.best_params_)
print(f"Control arm best CV RMSE: ${-gs_ctrl.best_score_:.4f}")

# Step 3 — Tuned T-Learner
treat_tuned = lgb.LGBMRegressor(
    n_estimators=200,
    verbose=-1,
    random_state=RANDOM_STATE,
    **gs_treat.best_params_
)

ctrl_tuned = lgb.LGBMRegressor(
    n_estimators=200,
    verbose=-1,
    random_state=RANDOM_STATE,
    **gs_ctrl.best_params_
)

treat_tuned.fit(X_treat, y_treat)
ctrl_tuned.fit(X_ctrl, y_ctrl)

mu1_tuned = treat_tuned.predict(Xf_te)
mu0_tuned = ctrl_tuned.predict(Xf_te)

cate_t_tuned = mu1_tuned - mu0_tuned

print(f"\nT-Learner tuned -- mean: ${cate_t_tuned.mean():.3f}  std: ${cate_t_tuned.std():.3f}")

# Step 4 — Compare distributions
summary_t = pd.DataFrame({
    "model": ["T-Learner naive", "T-Learner tuned"],
    "mean_CATE": [cate_t_naive.mean(), cate_t_tuned.mean()],
    "std_CATE": [cate_t_naive.std(), cate_t_tuned.std()],
    "share_above_break_even": [
        np.mean(cate_t_naive > BREAK_EVEN),
        np.mean(cate_t_tuned > BREAK_EVEN)
    ]
})

print("\nT-Learner CATE summary:")
print(summary_t)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(cate_t_naive, bins=50, alpha=0.6, label="Naive")
ax.hist(cate_t_tuned, bins=50, alpha=0.6, label="Tuned")

ax.axvline(cate_t_naive.mean(), linestyle="--", linewidth=1.5, label="Naive mean")
ax.axvline(cate_t_tuned.mean(), linestyle="--", linewidth=1.5, label="Tuned mean")
ax.axvline(BREAK_EVEN, linestyle=":", linewidth=2, label=f"Break-even = ${BREAK_EVEN:.2f}")

ax.set_title("T-Learner CATE Distribution")
ax.set_xlabel("Predicted CATE ($)")
ax.set_ylabel("Number of customers")
ax.legend()

plt.tight_layout()
plt.show()

# Required output checks
assert isinstance(cate_t_naive, np.ndarray)
assert isinstance(cate_t_tuned, np.ndarray)
assert cate_t_naive.shape == (len(Xf_te),)
assert cate_t_tuned.shape == (len(Xf_te),)

print("\nRequired outputs created:")
print("cate_t_naive shape:", cate_t_naive.shape)
print("cate_t_tuned shape:", cate_t_tuned.shape)

*Your answer:*The treated and control arms converged to the same best hyperparameters: learning_rate = 0.1, max_depth = 6, min_child_samples = 20, and num_leaves = 60. However, the treated arm has a much higher CV RMSE than the control arm, which suggests that spending among treated customers is more variable or heavier-tailed. Since both arms selected the largest `num_leaves` in the grid, the model needed relatively flexible trees for both conditional spending surfaces, even though the treated surface appears harder to fit.

---
## Part 2: Causal Forest (20 points)

### Background: `econml.grf.CausalForest`

`CausalForest` is econml's implementation of the Wager & Athey (2018) honest causal forest. Key features:
- **Honesty** (`honest=True` by default): splits chosen on one subsample, leaf estimates computed on a separate subsample, preventing the winner's curse
- **No built-in `.tune()`**: unlike `CausalForestDML`, plain `CausalForest` has no `.tune()` method. Tune manually via cross-validated R-loss over a hyperparameter grid

**Why R-loss, not MSE?** The true CATE $\tau(x_i)$ is never observed, so held-out MSE on $\hat{\tau}$ is not computable. The R-loss uses only observable quantities:

$$\mathcal{L}_{\text{R}}(\hat{\tau}) = \frac{1}{n}\sum_i \left(\tilde{Y}_i - \hat{\tau}(X_i)\tilde{D}_i\right)^2$$

where $\tilde{Y}$ and $\tilde{D}$ are cross-fitted residuals. We compute this via $k$-fold CV: nuisances are fit on training folds and the R-loss is evaluated on the held-out fold.

### 2a. Naive and Tuned Causal Forest (20 points)

**Step 1 — Naive.** Fit `CausalForest(**NAIVE_CF)` on the training set using the positional API: `cf.fit(Xf_tr, D_tr, Y_tr)`. Store test-set CATE as `cate_cf_naive`. Note: `predict(X)` returns shape `(n, 1)` — always call `.flatten()`.

**Step 2 — Tune.** Implement `r_loss_cv(Xf, Y, D, params, n_splits=3)` that:
1. Splits the training data into `n_splits` folds
2. In each fold, fits LightGBM nuisances on the training folds and predicts residuals $\tilde{Y}$, $\tilde{D}$ on the held-out fold
3. Fits a `CausalForest` on the training folds and predicts CATE on the held-out fold
4. Returns the mean R-loss across folds: $\frac{1}{n}\sum_i(\tilde{Y}_i - \hat{\tau}(X_i)\tilde{D}_i)^2$

Search over the grid: `min_samples_leaf` in {5, 10, 20}, `max_depth` in {None, 5}, `max_features` in {'auto', 0.5}. Print best params and best R-loss.

**Step 3 — Tuned model.** Refit `CausalForest` with best params and `n_estimators=1000`. Store as `cate_cf_tuned`.

**Step 4 — Compare.** Print mean and std of both. Plot overlapping histograms of naive vs. tuned CATE distributions.

✍️ **Written answer** (2–3 sentences): What hyperparameter changed most between the naive and tuned forest, and what does it control in the bias-variance tradeoff? Why is cross-validated R-loss a better tuning criterion for the causal forest than cross-validated RMSE on $Y$?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# cate_cf_naive  = np.array(...)    ← required
# cate_cf_tuned  = np.array(...)    ← required
# ── Part 2: CausalForest Naive and Tuned ─────────────────────────────────────
from econml.grf import CausalForest
from sklearn.model_selection import KFold
import itertools

# Step 1 — Naive Causal Forest
cf_naive = CausalForest(**NAIVE_CF)
cf_naive.fit(Xf_tr, D_tr, Y_tr)

cate_cf_naive = cf_naive.predict(Xf_te).flatten()

print(f"CausalForest naive -- mean: ${cate_cf_naive.mean():.3f}  std: ${cate_cf_naive.std():.3f}")
print("cate_cf_naive shape:", cate_cf_naive.shape)


# Step 2 — Tune by cross-validated R-loss
CF_GRID = [
    {
        "min_samples_leaf": min_leaf,
        "max_depth": max_depth,
        "max_features": max_features
    }
    for min_leaf, max_depth, max_features in itertools.product(
        [5, 10, 20],
        [None, 5],
        ["auto", 0.5]
    )
]

def r_loss_cv(Xf, Y, D, params, n_splits=3, random_state=RANDOM_STATE):
    """
    Cross-validated R-loss for tuning CausalForest.

    For each fold:
      1. Fit nuisance models E[Y|X] and E[D|X] on training folds.
      2. Compute residuals Y_tilde and D_tilde on held-out fold.
      3. Fit CausalForest on training folds.
      4. Predict tau_hat on held-out fold.
      5. Evaluate mean (Y_tilde - tau_hat * D_tilde)^2.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_losses = []

    for fold, (idx_train, idx_val) in enumerate(kf.split(Xf), start=1):
        X_train, X_val = Xf[idx_train], Xf[idx_val]
        Y_train, Y_val = Y[idx_train], Y[idx_val]
        D_train, D_val = D[idx_train], D[idx_val]

        # Nuisance models for residualization
        m_y = lgb.LGBMRegressor(
            n_estimators=100,
            max_depth=4,
            num_leaves=31,
            learning_rate=0.05,
            min_child_samples=20,
            verbose=-1,
            random_state=random_state
        )

        m_t = lgb.LGBMClassifier(
            n_estimators=100,
            max_depth=4,
            num_leaves=31,
            learning_rate=0.05,
            min_child_samples=20,
            verbose=-1,
            random_state=random_state
        )

        m_y.fit(X_train, Y_train)
        m_t.fit(X_train, D_train)

        Y_tilde = Y_val - m_y.predict(X_val)
        D_tilde = D_val - m_t.predict_proba(X_val)[:, 1]

        # Causal forest on training folds
        cf = CausalForest(
            n_estimators=200,
            honest=True,
            random_state=random_state,
            **params
        )

        cf.fit(X_train, D_train, Y_train)
        tau_hat = cf.predict(X_val).flatten()

        fold_loss = np.mean((Y_tilde - tau_hat * D_tilde) ** 2)
        fold_losses.append(fold_loss)

    return float(np.mean(fold_losses))


best_loss = np.inf
best_params = None
r_loss_results = []

print("Tuning CausalForest by cross-validated R-loss...")

for params in CF_GRID:
    loss = r_loss_cv(Xf_tr, Y_tr, D_tr, params, n_splits=3)
    r_loss_results.append({**params, "r_loss": loss})

    print(f"params={params}  R-loss={loss:.6f}")

    if loss < best_loss:
        best_loss = loss
        best_params = params

print("\nBest CausalForest params:")
print(best_params)
print(f"Best CV R-loss: {best_loss:.6f}")

r_loss_df = pd.DataFrame(r_loss_results).sort_values("r_loss")
display(r_loss_df)


# Step 3 — Tuned Causal Forest
cf_tuned = CausalForest(
    n_estimators=1000,
    honest=True,
    random_state=RANDOM_STATE,
    **best_params
)

cf_tuned.fit(Xf_tr, D_tr, Y_tr)

cate_cf_tuned = cf_tuned.predict(Xf_te).flatten()

print(f"\nCausalForest tuned -- mean: ${cate_cf_tuned.mean():.3f}  std: ${cate_cf_tuned.std():.3f}")
print("cate_cf_tuned shape:", cate_cf_tuned.shape)


# Step 4 — Compare
summary_cf = pd.DataFrame({
    "model": ["CausalForest naive", "CausalForest tuned"],
    "mean_CATE": [cate_cf_naive.mean(), cate_cf_tuned.mean()],
    "std_CATE": [cate_cf_naive.std(), cate_cf_tuned.std()],
    "share_above_break_even": [
        np.mean(cate_cf_naive > BREAK_EVEN),
        np.mean(cate_cf_tuned > BREAK_EVEN)
    ]
})

print("\nCausalForest CATE summary:")
print(summary_cf)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(cate_cf_naive, bins=50, alpha=0.6, label="Naive")
ax.hist(cate_cf_tuned, bins=50, alpha=0.6, label="Tuned")

ax.axvline(cate_cf_naive.mean(), linestyle="--", linewidth=1.5, label="Naive mean")
ax.axvline(cate_cf_tuned.mean(), linestyle="--", linewidth=1.5, label="Tuned mean")
ax.axvline(BREAK_EVEN, linestyle=":", linewidth=2, label=f"Break-even = ${BREAK_EVEN:.2f}")

ax.set_title("CausalForest CATE Distribution")
ax.set_xlabel("Predicted CATE ($)")
ax.set_ylabel("Number of customers")
ax.legend()

plt.tight_layout()
plt.show()

# Required output checks
assert isinstance(cate_cf_naive, np.ndarray)
assert isinstance(cate_cf_tuned, np.ndarray)
assert cate_cf_naive.shape == (len(Xf_te),)
assert cate_cf_tuned.shape == (len(Xf_te),)

print("\nRequired outputs created:")
print("cate_cf_naive shape:", cate_cf_naive.shape)
print("cate_cf_tuned shape:", cate_cf_tuned.shape)

*Your answer:*The tuned causal forest changed mainly by reducing `min_samples_leaf` from 10 to 5 and using `max_features = 0.5` instead of the naive setting. A smaller `min_samples_leaf` lets the forest create more granular leaves, which can capture more treatment-effect heterogeneity but may increase variance; using only a subset of features at each split can also reduce correlation across trees. Cross-validated R-loss is a better tuning criterion than RMSE on Y because the true CATE is never directly observed, and R-loss evaluates whether the estimated treatment effects explain residualized outcome variation after removing baseline outcome and treatment-propensity components.

---
## Part 3: DML Family (30 points)

### Background: Three DML Final Stages

All three DML models share the same **first stage**: estimate nuisance functions $\hat{\ell}(X) = E[Y|X]$ (`model_y`) and $\hat{r}(X) = E[D|X]$ (`model_t`) via cross-fitting, then compute residuals $\tilde{Y} = Y - \hat{\ell}(X)$ and $\tilde{D} = D - \hat{r}(X)$. They differ only in the **final stage** — how they use those residuals to estimate $\tau(x)$:

| Model | Final stage | CATE assumption |
|-------|-------------|----------------|
| `LinearDML` | OLS of $\tilde{Y}$ on $\tilde{D} \cdot X$ | $\tau(x) = \theta^\top x$ — linear in features, in dollars |
| `NonParamDML` | Any ML model mapping $(\tilde{Y}/\tilde{D},\ X) \to \tau$ with weights $\tilde{D}^2$ | $\tau(x)$ fully flexible, in dollars |
| `CausalForestDML` | Honest causal forest on residuals | $\tau(x)$ via forest, in dollars |

Because all three share the first stage, **tuning the nuisances is shared work**: tune `model_y` and `model_t` once, then plug the same tuned models into all three DML variants.

**Tuning rationale for nuisances.** The nuisance models are pure supervised problems:
- `model_y`: predict spending $Y$ from $X$ — a **regression** problem. Use `lgb.LGBMRegressor`, tune by **cross-validated MSE** (`neg_mean_squared_error`).
- `model_t`: predict $D$ from $X$ — a **classification** problem. Use `lgb.LGBMClassifier`, tune by **cross-validated log-loss**.

### 3a. Tune the Shared Nuisance Models (8 points)

**Step 1 — Naive nuisances.** Define `model_y_naive = lgb.LGBMRegressor(**NAIVE_LGB)` and `model_t_naive = lgb.LGBMClassifier(**NAIVE_LGB)`.

**Step 2 — Tune `model_y`.** Run `GridSearchCV` (5-fold, `neg_mean_squared_error`) on `(Xs_tr, Y_tr)`. Print best params and best CV MSE.

**Step 3 — Tune `model_t`.** Run `GridSearchCV` (5-fold, `neg_log_loss`) on `(Xs_tr, D_tr)`. Print best params and best CV log-loss.

**Step 4 — Cross-fit residuals.** Using the tuned models, run a 5-fold cross-fitting loop to compute out-of-fold residuals $\tilde{Y}$ and $\tilde{D}$ on the full training set. Save `Y_tilde_tr` and `D_tilde_tr` — you will need them in Part 3c.

**Report:** a summary table with one row per nuisance model (naive vs. tuned) showing CV MSE for `model_y` and CV log-loss for `model_t`.

✍️ **Written answer** (2–3 sentences): Does better nuisance fit (lower MSE / log-loss) guarantee a better final CATE estimate — why or why not? In which of the three DML final stages does nuisance quality matter most?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
#
# model_y uses lgb.LGBMRegressor — predicting continuous spending Y from X
# model_t uses lgb.LGBMClassifier — predicting binary treatment D from X
#
# Tune model_y by CV MSE (neg_mean_squared_error)
# Tune model_t by CV log-loss (neg_log_loss)
#
# Save for Part 3b:
# Y_tilde_tr = np.array(...)    ← cross-fitted Y residuals on training set
# D_tilde_tr = np.array(...)    ← cross-fitted D residuals on training set
#
# model_y_naive  = lgb.LGBMRegressor(**NAIVE_LGB)
# model_t_naive  = lgb.LGBMClassifier(**NAIVE_LGB)
# model_y_tuned  = lgb.LGBMRegressor(...)   ← best params
# model_t_tuned  = lgb.LGBMClassifier(...)  ← best params

# ── Part 3a: Tune Shared Nuisance Models ─────────────────────────────────────
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.base import clone

# Step 1 — Naive nuisance models
model_y_naive = lgb.LGBMRegressor(**NAIVE_LGB)
model_t_naive = lgb.LGBMClassifier(**NAIVE_LGB)

cv5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Naive CV metrics
naive_y_mse_scores = cross_val_score(
    model_y_naive,
    Xs_tr,
    Y_tr,
    scoring="neg_mean_squared_error",
    cv=cv5,
    n_jobs=-1
)

naive_t_logloss_scores = cross_val_score(
    model_t_naive,
    Xs_tr,
    D_tr,
    scoring="neg_log_loss",
    cv=cv5,
    n_jobs=-1
)

naive_y_mse = -naive_y_mse_scores.mean()
naive_t_logloss = -naive_t_logloss_scores.mean()

print(f"Naive model_y CV MSE: {naive_y_mse:.6f}")
print(f"Naive model_t CV log-loss: {naive_t_logloss:.6f}")


# Step 2 — Tune model_y by CV MSE
gs_y = GridSearchCV(
    estimator=lgb.LGBMRegressor(
        n_estimators=200,
        verbose=-1,
        random_state=RANDOM_STATE
    ),
    param_grid=LGB_GRID,
    scoring="neg_mean_squared_error",
    cv=cv5,
    n_jobs=-1
)

gs_y.fit(Xs_tr, Y_tr)

best_y_mse = -gs_y.best_score_

print("\nTuned model_y best params:")
print(gs_y.best_params_)
print(f"Tuned model_y best CV MSE: {best_y_mse:.6f}")


# Step 3 — Tune model_t by CV log-loss
gs_t = GridSearchCV(
    estimator=lgb.LGBMClassifier(
        n_estimators=200,
        verbose=-1,
        random_state=RANDOM_STATE
    ),
    param_grid=LGB_GRID,
    scoring="neg_log_loss",
    cv=cv5,
    n_jobs=-1
)

gs_t.fit(Xs_tr, D_tr)

best_t_logloss = -gs_t.best_score_

print("\nTuned model_t best params:")
print(gs_t.best_params_)
print(f"Tuned model_t best CV log-loss: {best_t_logloss:.6f}")


# Define tuned nuisance models for later DML parts
model_y_tuned = lgb.LGBMRegressor(
    n_estimators=200,
    verbose=-1,
    random_state=RANDOM_STATE,
    **gs_y.best_params_
)

model_t_tuned = lgb.LGBMClassifier(
    n_estimators=200,
    verbose=-1,
    random_state=RANDOM_STATE,
    **gs_t.best_params_
)


# Step 4 — Cross-fit residuals on the training set using tuned nuisance models
Y_tilde_tr = np.zeros_like(Y_tr, dtype=float)
D_tilde_tr = np.zeros_like(D_tr, dtype=float)

for fold, (train_idx, val_idx) in enumerate(cv5.split(Xs_tr), start=1):
    X_train, X_val = Xs_tr[train_idx], Xs_tr[val_idx]
    Y_train, Y_val = Y_tr[train_idx], Y_tr[val_idx]
    D_train, D_val = D_tr[train_idx], D_tr[val_idx]

    my_fold = clone(model_y_tuned)
    mt_fold = clone(model_t_tuned)

    my_fold.fit(X_train, Y_train)
    mt_fold.fit(X_train, D_train)

    y_hat_val = my_fold.predict(X_val)
    d_hat_val = mt_fold.predict_proba(X_val)[:, 1]

    Y_tilde_tr[val_idx] = Y_val - y_hat_val
    D_tilde_tr[val_idx] = D_val - d_hat_val

    print(
        f"Fold {fold}: "
        f"mean(Y_tilde)={Y_tilde_tr[val_idx].mean():.4f}, "
        f"mean(D_tilde)={D_tilde_tr[val_idx].mean():.4f}"
    )


# Summary table
nuisance_summary = pd.DataFrame({
    "version": ["naive", "tuned"],
    "model_y_CV_MSE": [naive_y_mse, best_y_mse],
    "model_t_CV_log_loss": [naive_t_logloss, best_t_logloss]
})

print("\nNuisance model CV summary:")
display(nuisance_summary)

print("\nCross-fitted residuals created:")
print("Y_tilde_tr shape:", Y_tilde_tr.shape)
print("D_tilde_tr shape:", D_tilde_tr.shape)
print("Y_tilde_tr mean:", Y_tilde_tr.mean())
print("D_tilde_tr mean:", D_tilde_tr.mean())

# Required checks
assert isinstance(Y_tilde_tr, np.ndarray)
assert isinstance(D_tilde_tr, np.ndarray)
assert Y_tilde_tr.shape == Y_tr.shape
assert D_tilde_tr.shape == D_tr.shape

*Your answer:*The tuned nuisance models improved the supervised first-stage fit, reducing model_y CV MSE from about 82.64 to 77.79 and model_t CV log-loss from about 0.677 to 0.631. However, better nuisance fit does not guarantee a better final CATE estimate, because the final target is treatment-effect heterogeneity rather than prediction of Y or D alone. Nuisance quality matters most for the flexible final stages, especially NonParamDML/R-learner and CausalForestDML, because noisy residuals can be amplified when the final model has high flexibility.


`LinearDML` assumes $\tau(x) = \theta^\top x$ — the treatment effect is linear in features, measured in dollars. Despite this restriction, it may perform well if the true heterogeneity is approximately linear, and it gives directly interpretable coefficients: each $\theta_j$ tells you how the treatment effect changes per unit of feature $j$.

**Step 1 — Naive.** Instantiate `LinearDML(model_y=model_y_naive, model_t=model_t_naive, discrete_treatment=True, cv=5, random_state=RANDOM_STATE)`. Call `.fit(Y_tr, D_tr, X=Xs_tr)`. Store test-set CATE as `cate_ldml_naive`.

**Step 2 — Tuned.** Same but with `model_y_tuned` and `model_t_tuned`. Store as `cate_ldml_tuned`.

**Step 3 — Coefficients.** From the tuned model, retrieve the CATE coefficient table using `.coef__inference()`. Note: `.pvalue` is a method — call it as `.pvalue()`. Print coefficients, standard errors, p-values, and 95% CIs. Which features have statistically significant CATE coefficients at the 5% level?

**Step 4 — Compare.** Plot overlapping histograms of `cate_ldml_naive` and `cate_ldml_tuned`.

✍️ **Written answer** (2–3 sentences): How do the significant CATE coefficients in `LinearDML` compare to the significant interaction terms in the OLS model from Part 1a? If they agree, what does that suggest about the linearity of the true CATE in this dataset?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# cate_ldml_naive  = np.array(...)    ← required
# cate_ldml_tuned  = np.array(...)    ← required
# ── Part 3b: LinearDML Naive and Tuned ───────────────────────────────────────
from econml.dml import LinearDML

# Step 1 — Naive LinearDML
ldml_naive = LinearDML(
    model_y=model_y_naive,
    model_t=model_t_naive,
    discrete_treatment=True,
    cv=5,
    random_state=RANDOM_STATE
)

ldml_naive.fit(Y_tr, D_tr, X=Xs_tr)

cate_ldml_naive = ldml_naive.effect(Xs_te).flatten()

print(f"LinearDML naive -- mean: ${cate_ldml_naive.mean():.3f}  std: ${cate_ldml_naive.std():.3f}")


# Step 2 — Tuned LinearDML
ldml_tuned = LinearDML(
    model_y=model_y_tuned,
    model_t=model_t_tuned,
    discrete_treatment=True,
    cv=5,
    random_state=RANDOM_STATE
)

ldml_tuned.fit(Y_tr, D_tr, X=Xs_tr)

cate_ldml_tuned = ldml_tuned.effect(Xs_te).flatten()

print(f"LinearDML tuned -- mean: ${cate_ldml_tuned.mean():.3f}  std: ${cate_ldml_tuned.std():.3f}")


# Step 3 — Coefficient table from tuned LinearDML
# Note: pvalue is a method, so use pvalue()
inf = ldml_tuned.coef__inference()

coef_table_ldml = pd.DataFrame({
    "coef": ldml_tuned.coef_.flatten(),
    "std_err": inf.stderr.flatten(),
    "pvalue": inf.pvalue().flatten(),
    "ci_lower": inf.conf_int()[0].flatten(),
    "ci_upper": inf.conf_int()[1].flatten()
}, index=FEATURE_COLS)

print("\nLinearDML CATE coefficients, tuned model:")
display(coef_table_ldml.round(4))

sig_ldml = coef_table_ldml[coef_table_ldml["pvalue"] < 0.05]

print("\nSignificant LinearDML CATE coefficients at 5%:")
print(list(sig_ldml.index))


# Step 4 — Compare distributions
summary_ldml = pd.DataFrame({
    "model": ["LinearDML naive", "LinearDML tuned"],
    "mean_CATE": [cate_ldml_naive.mean(), cate_ldml_tuned.mean()],
    "std_CATE": [cate_ldml_naive.std(), cate_ldml_tuned.std()],
    "share_above_break_even": [
        np.mean(cate_ldml_naive > BREAK_EVEN),
        np.mean(cate_ldml_tuned > BREAK_EVEN)
    ]
})

print("\nLinearDML CATE summary:")
display(summary_ldml)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(cate_ldml_naive, bins=50, alpha=0.6, label="Naive")
ax.hist(cate_ldml_tuned, bins=50, alpha=0.6, label="Tuned")

ax.axvline(cate_ldml_naive.mean(), linestyle="--", linewidth=1.5, label="Naive mean")
ax.axvline(cate_ldml_tuned.mean(), linestyle="--", linewidth=1.5, label="Tuned mean")
ax.axvline(BREAK_EVEN, linestyle=":", linewidth=2, label=f"Break-even = ${BREAK_EVEN:.2f}")

ax.set_title("LinearDML CATE Distribution")
ax.set_xlabel("Predicted CATE ($)")
ax.set_ylabel("Number of customers")
ax.legend()

plt.tight_layout()
plt.show()


# Required output checks
assert isinstance(cate_ldml_naive, np.ndarray)
assert isinstance(cate_ldml_tuned, np.ndarray)
assert cate_ldml_naive.shape == (len(Xs_te),)
assert cate_ldml_tuned.shape == (len(Xs_te),)

print("\nRequired outputs created:")
print("cate_ldml_naive shape:", cate_ldml_naive.shape)
print("cate_ldml_tuned shape:", cate_ldml_tuned.shape)

*Your answer:*The significant LinearDML CATE coefficients are gender_F, gender_O, and missing_demo, which mostly agree with the OLS interaction model from Part 1a where gender_F:D, gender_O:D, and missing_demo:D were also significant. However, OLS also found gender_M:D significant, while LinearDML does not find gender_M significant after residualizing the outcome and treatment. Overall, this suggests that some of the treatment-effect heterogeneity in this dataset is approximately linear in the demographic features, especially gender and missing-demographic status, but not every OLS interaction remains robust under the DML approach.

### 3c. NonParamDML (R-Learner): Naive and Tuned (8 points)

`NonParamDML` is the general R-Learner: after residualizing, it fits any ML model as the final stage. You will use a LightGBM regressor as the final stage. The model solves:

$$\hat{\tau}(\cdot) = \arg\min_{\theta} \sum_i \left(\tilde{Y}_i - \theta(X_i)\tilde{D}_i\right)^2$$

which is equivalent to a weighted regression of $\tilde{Y}_i / \tilde{D}_i$ on $X_i$ with weights $\tilde{D}_i^2$ (econml handles this internally).

**Step 1 — Naive.** Instantiate:
```python
NonParamDML(
    model_y=model_y_naive,
    model_t=model_t_naive,
    model_final=lgb.LGBMRegressor(**NAIVE_LGB),
    discrete_treatment=True, cv=5, random_state=RANDOM_STATE
)
```
Call `.fit(Y_tr, D_tr, X=Xs_tr)`. Store test-set CATE as `cate_rl_naive`.

**Step 2 — Tuned.** Use `model_y_tuned` and `model_t_tuned` for the nuisances. For the final stage, tune a `lgb.LGBMRegressor` by cross-validated **MSE** on the pseudo-outcome $\tilde{Y}/\tilde{D}$ (using `Y_tilde_tr` and `D_tilde_tr` from Part 3a, clipping $|\tilde{D}|<0.05$) with sample weights $\tilde{D}^2$. Note: in sklearn 1.6+, pass `sample_weight` directly to `.fit()`, not to the `GridSearchCV` constructor. Print best final-stage params. Store as `cate_rl_tuned`.

**Step 3 — Compare.** Plot overlapping histograms of `cate_rl_naive` and `cate_rl_tuned`.

✍️ **Written answer** (2–3 sentences): The final-stage tuning uses MSE on the pseudo-outcome $\tilde{Y}/\tilde{D}$ — a standard supervised problem. How does this differ from the cross-validated R-loss used to tune the causal forest in Part 2, and is it equally valid as a tuning criterion for the R-Learner final stage?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# cate_rl_naive  = np.array(...)    ← required
# cate_rl_tuned  = np.array(...)    ← required

# ── Part 3c: NonParamDML / R-Learner Naive and Tuned ─────────────────────────
from econml.dml import NonParamDML

# Step 1 — Naive NonParamDML / R-Learner
rl_naive = NonParamDML(
    model_y=model_y_naive,
    model_t=model_t_naive,
    model_final=lgb.LGBMRegressor(**NAIVE_LGB),
    discrete_treatment=True,
    cv=5,
    random_state=RANDOM_STATE
)

rl_naive.fit(Y_tr, D_tr, X=Xs_tr)

cate_rl_naive = rl_naive.effect(Xs_te).flatten()

print(f"NonParamDML naive -- mean: ${cate_rl_naive.mean():.3f}  std: ${cate_rl_naive.std():.3f}")


# Step 2 — Tune final-stage LightGBM on pseudo-outcome Y_tilde / D_tilde
eps = 0.05

# Clip small D residuals to avoid exploding pseudo-outcomes
D_tilde_clip = np.where(
    np.abs(D_tilde_tr) < eps,
    eps * np.where(D_tilde_tr >= 0, 1, -1),
    D_tilde_tr
)

pseudo_y = Y_tilde_tr / D_tilde_clip
pseudo_w = D_tilde_tr ** 2

print("\nPseudo-outcome summary:")
print(pd.Series(pseudo_y).describe())
print("\nPseudo-weight summary:")
print(pd.Series(pseudo_w).describe())

gs_rl_final = GridSearchCV(
    estimator=lgb.LGBMRegressor(
        n_estimators=200,
        verbose=-1,
        random_state=RANDOM_STATE
    ),
    param_grid=LGB_GRID,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

# In sklearn 1.6+, pass sample_weight directly to fit()
gs_rl_final.fit(Xs_tr, pseudo_y, sample_weight=pseudo_w)

print("\nR-Learner final-stage best params:")
print(gs_rl_final.best_params_)
print(f"Best final-stage CV MSE on pseudo-outcome: {-gs_rl_final.best_score_:.6f}")

rl_final_tuned = lgb.LGBMRegressor(
    n_estimators=200,
    verbose=-1,
    random_state=RANDOM_STATE,
    **gs_rl_final.best_params_
)

# Tuned NonParamDML
rl_tuned = NonParamDML(
    model_y=model_y_tuned,
    model_t=model_t_tuned,
    model_final=rl_final_tuned,
    discrete_treatment=True,
    cv=5,
    random_state=RANDOM_STATE
)

rl_tuned.fit(Y_tr, D_tr, X=Xs_tr)

cate_rl_tuned = rl_tuned.effect(Xs_te).flatten()

print(f"\nNonParamDML tuned -- mean: ${cate_rl_tuned.mean():.3f}  std: ${cate_rl_tuned.std():.3f}")


# Step 3 — Compare
summary_rl = pd.DataFrame({
    "model": ["NonParamDML naive", "NonParamDML tuned"],
    "mean_CATE": [cate_rl_naive.mean(), cate_rl_tuned.mean()],
    "std_CATE": [cate_rl_naive.std(), cate_rl_tuned.std()],
    "share_above_break_even": [
        np.mean(cate_rl_naive > BREAK_EVEN),
        np.mean(cate_rl_tuned > BREAK_EVEN)
    ]
})

print("\nNonParamDML / R-Learner CATE summary:")
display(summary_rl)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(cate_rl_naive, bins=50, alpha=0.6, label="Naive")
ax.hist(cate_rl_tuned, bins=50, alpha=0.6, label="Tuned")

ax.axvline(cate_rl_naive.mean(), linestyle="--", linewidth=1.5, label="Naive mean")
ax.axvline(cate_rl_tuned.mean(), linestyle="--", linewidth=1.5, label="Tuned mean")
ax.axvline(BREAK_EVEN, linestyle=":", linewidth=2, label=f"Break-even = ${BREAK_EVEN:.2f}")

ax.set_title("NonParamDML / R-Learner CATE Distribution")
ax.set_xlabel("Predicted CATE ($)")
ax.set_ylabel("Number of customers")
ax.legend()

plt.tight_layout()
plt.show()


# Required output checks
assert isinstance(cate_rl_naive, np.ndarray)
assert isinstance(cate_rl_tuned, np.ndarray)
assert cate_rl_naive.shape == (len(Xs_te),)
assert cate_rl_tuned.shape == (len(Xs_te),)

print("\nRequired outputs created:")
print("cate_rl_naive shape:", cate_rl_naive.shape)
print("cate_rl_tuned shape:", cate_rl_tuned.shape)

*Your answer:*The final-stage tuning in the R-Learner treats the problem as a supervised weighted regression of the pseudo-outcome \(\tilde{Y}/\tilde{D}\) on \(X\), with weights \(\tilde{D}^2\). This differs from the causal forest tuning in Part 2, where cross-validated R-loss directly evaluates how well \(\hat{\tau}(X)\tilde{D}\) explains the residualized outcome \(\tilde{Y}\) on held-out folds. It is valid for the R-Learner final stage because the weighted pseudo-outcome regression is algebraically equivalent to the R-loss objective, but it can be noisy when \(\tilde{D}\) is small, which is why clipping is used.

### 3d. CausalForestDML (Forest-DML): Naive and Tuned (8 points)

`CausalForestDML` combines the DML first stage with an honest causal forest final stage. It is the most flexible of the three DML variants: the forest discovers nonlinear, nonparametric heterogeneity in the residualized space.

The key difference from plain `CausalForest` (Part 2) is that `CausalForestDML` **separates** nuisance estimation from the forest — you plug in any `model_y` and `model_t`. Plain `CausalForest` estimates its own nuisances internally. This separation means nuisance quality directly affects the final CATE estimates. Also note: `n_estimators` must be divisible by `subforest_size` (default 4) — use `n_estimators=1000`.

**Step 1 — Naive.** Instantiate:
```python
CausalForestDML(
    model_y=model_y_naive,
    model_t=model_t_naive,
    discrete_treatment=True, cv=5,
    n_estimators=1000, min_samples_leaf=10,
    random_state=RANDOM_STATE
)
```
Call `.fit(Y_tr, D_tr, X=Xs_tr)`. Store test-set CATE as `cate_cfdml_naive`.

**Step 2 — Tuned.** Use `model_y_tuned` and `model_t_tuned`. Then call `.tune(Y_tr, D_tr, X=Xs_tr)` on the `CausalForestDML` instance to tune the forest hyperparameters by OOB R-loss. Print the tuned forest params. Then call `.fit(Y_tr, D_tr, X=Xs_tr)`. Store as `cate_cfdml_tuned`.

**Step 3 — Compare.** Plot overlapping histograms of `cate_cfdml_naive` and `cate_cfdml_tuned`.

✍️ **Written answer** (2–3 sentences): `CausalForestDML` undergoes two rounds of tuning — supervised MSE/log-loss for the nuisances, then OOB R-loss for the forest. Why are different criteria used for each stage? How does `CausalForestDML` differ architecturally from plain `CausalForest`, and when would you prefer one over the other?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# cate_cfdml_naive  = np.array(...)    ← required
# cate_cfdml_tuned  = np.array(...)    ← required
# ── Part 3d: CausalForestDML Naive and Tuned ─────────────────────────────────
from econml.dml import CausalForestDML

# Step 1 — Naive CausalForestDML
cfdml_naive = CausalForestDML(
    model_y=model_y_naive,
    model_t=model_t_naive,
    discrete_treatment=True,
    cv=5,
    n_estimators=1000,        # divisible by subforest_size=4
    min_samples_leaf=10,
    random_state=RANDOM_STATE
)

cfdml_naive.fit(Y_tr, D_tr, X=Xs_tr)

cate_cfdml_naive = cfdml_naive.effect(Xs_te).flatten()

print(f"CausalForestDML naive -- mean: ${cate_cfdml_naive.mean():.3f}  std: ${cate_cfdml_naive.std():.3f}")


# Step 2 — Tuned CausalForestDML
cfdml_tuned = CausalForestDML(
    model_y=model_y_tuned,
    model_t=model_t_tuned,
    discrete_treatment=True,
    cv=5,
    n_estimators=1000,        # must be divisible by subforest_size=4
    min_samples_leaf=10,
    random_state=RANDOM_STATE
)

# Tune forest hyperparameters by OOB R-loss
cfdml_tuned.tune(Y_tr, D_tr, X=Xs_tr)

print("\nTuned CausalForestDML forest parameters:")
print("min_samples_leaf:", getattr(cfdml_tuned, "min_samples_leaf", None))
print("max_depth:", getattr(cfdml_tuned, "max_depth", None))
print("max_features:", getattr(cfdml_tuned, "max_features", None))
print("min_balancedness_tol:", getattr(cfdml_tuned, "min_balancedness_tol", None))

# Fit tuned model
cfdml_tuned.fit(Y_tr, D_tr, X=Xs_tr)

cate_cfdml_tuned = cfdml_tuned.effect(Xs_te).flatten()

print(f"\nCausalForestDML tuned -- mean: ${cate_cfdml_tuned.mean():.3f}  std: ${cate_cfdml_tuned.std():.3f}")


# Step 3 — Compare
summary_cfdml = pd.DataFrame({
    "model": ["CausalForestDML naive", "CausalForestDML tuned"],
    "mean_CATE": [cate_cfdml_naive.mean(), cate_cfdml_tuned.mean()],
    "std_CATE": [cate_cfdml_naive.std(), cate_cfdml_tuned.std()],
    "share_above_break_even": [
        np.mean(cate_cfdml_naive > BREAK_EVEN),
        np.mean(cate_cfdml_tuned > BREAK_EVEN)
    ]
})

print("\nCausalForestDML CATE summary:")
display(summary_cfdml)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(cate_cfdml_naive, bins=50, alpha=0.6, label="Naive")
ax.hist(cate_cfdml_tuned, bins=50, alpha=0.6, label="Tuned")

ax.axvline(cate_cfdml_naive.mean(), linestyle="--", linewidth=1.5, label="Naive mean")
ax.axvline(cate_cfdml_tuned.mean(), linestyle="--", linewidth=1.5, label="Tuned mean")
ax.axvline(BREAK_EVEN, linestyle=":", linewidth=2, label=f"Break-even = ${BREAK_EVEN:.2f}")

ax.set_title("CausalForestDML CATE Distribution")
ax.set_xlabel("Predicted CATE ($)")
ax.set_ylabel("Number of customers")
ax.legend()

plt.tight_layout()
plt.show()


# Required output checks
assert isinstance(cate_cfdml_naive, np.ndarray)
assert isinstance(cate_cfdml_tuned, np.ndarray)
assert cate_cfdml_naive.shape == (len(Xs_te),)
assert cate_cfdml_tuned.shape == (len(Xs_te),)

print("\nRequired outputs created:")
print("cate_cfdml_naive shape:", cate_cfdml_naive.shape)
print("cate_cfdml_tuned shape:", cate_cfdml_tuned.shape)

*Your answer:*Different criteria are used because the two stages solve different tasks. The nuisance models are standard supervised prediction problems, so MSE is appropriate for predicting spending \(Y\) and log-loss is appropriate for predicting binary treatment \(D\). The forest stage is estimating treatment-effect heterogeneity, so OOB R-loss is more appropriate because it evaluates how well the estimated CATE explains residualized outcome variation.

CausalForestDML differs from plain CausalForest because it separates the nuisance estimation stage from the forest and lets us plug in tuned \(E[Y|X]\) and \(E[D|X]\) models, while plain CausalForest handles nuisance estimation internally. I would prefer CausalForestDML when I want more control over the first-stage models or believe nuisance quality matters, and prefer plain CausalForest when I want a simpler causal forest pipeline with fewer moving parts.

---
# STAGE 2 — EVALUATE

Evaluate all 13 CATE vectors on the test set. This is the **only** stage where test-set outcomes are used for comparison.

---
## Part 4: Model Evaluation and Selection (25 points)

You now have 13 CATE arrays on the test set. Collect them in the `ALL_MODELS` dict before running Part 4a.

In [ ]:
# ── Populate this dict — do not change the keys ───────────────────────────────
ALL_MODELS = {
    # Linear benchmark
    'Linear (OLS)':          cate_linear,
    # Meta-learners
    'S-Learner naive':       cate_s_naive,
    'S-Learner tuned':       cate_s_tuned,
    'T-Learner naive':       cate_t_naive,
    'T-Learner tuned':       cate_t_tuned,
    # Causal forest
    'CausalForest naive':    cate_cf_naive,
    'CausalForest tuned':    cate_cf_tuned,
    # DML family
    'LinearDML naive':       cate_ldml_naive,
    'LinearDML tuned':       cate_ldml_tuned,
    'NonParamDML naive':     cate_rl_naive,
    'NonParamDML tuned':     cate_rl_tuned,
    'CausalForestDML naive': cate_cfdml_naive,
    'CausalForestDML tuned': cate_cfdml_tuned,
}
assert len(ALL_MODELS) == 13, "Expected 13 models"
print("ALL_MODELS populated:", list(ALL_MODELS.keys()))

### 4a. CATE Distribution Summary (5 points)

Produce a **summary table** — one row per model — with columns: Mean CATE (\$), Std CATE (\$), Min (\$), Max (\$), and Fraction above break-even ($\hat{\tau} > \$2$).

Then produce a **13-panel histogram figure** (3×5 grid, last two panels blank). In each panel include a vertical dashed red line at the mean and a vertical dotted gray line at the break-even threshold \\$2. Title each panel with the model name and the fraction above \\$2.

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# ── Part 4a: CATE Distribution Summary ──────────────────────────────────────

# Check that all 13 CATE arrays exist and have correct length
assert len(ALL_MODELS) == 13

for name, cate in ALL_MODELS.items():
    cate = np.asarray(cate)
    assert cate.shape == (len(Xs_te),), f"{name} has wrong shape: {cate.shape}"

# Summary table
rows = []

for name, cate in ALL_MODELS.items():
    cate = np.asarray(cate)

    rows.append({
        "Model": name,
        "Mean CATE ($)": cate.mean(),
        "Std CATE ($)": cate.std(),
        "Min ($)": cate.min(),
        "Max ($)": cate.max(),
        "Fraction above break-even": np.mean(cate > BREAK_EVEN)
    })

cate_dist_summary = pd.DataFrame(rows)

display(
    cate_dist_summary
    .sort_values("Mean CATE ($)", ascending=False)
    .style.format({
        "Mean CATE ($)": "{:.3f}",
        "Std CATE ($)": "{:.3f}",
        "Min ($)": "{:.3f}",
        "Max ($)": "{:.3f}",
        "Fraction above break-even": "{:.2%}"
    })
)

# 13-panel histogram figure: 3 x 5 grid, last two panels blank
fig, axes = plt.subplots(3, 5, figsize=(20, 11))
axes = axes.flatten()

for ax, (name, cate) in zip(axes, ALL_MODELS.items()):
    cate = np.asarray(cate)
    frac_above = np.mean(cate > BREAK_EVEN)

    ax.hist(cate, bins=50, alpha=0.75)

    # Required vertical lines
    ax.axvline(
        cate.mean(),
        color="red",
        linestyle="--",
        linewidth=1.5,
        label="Mean"
    )

    ax.axvline(
        BREAK_EVEN,
        color="gray",
        linestyle=":",
        linewidth=2,
        label="Break-even"
    )

    ax.set_title(f"{name}\nFraction > $2: {frac_above:.1%}", fontsize=10)
    ax.set_xlabel("Predicted CATE ($)")
    ax.set_ylabel("Count")

# Blank out remaining two panels
for j in range(len(ALL_MODELS), len(axes)):
    axes[j].axis("off")

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")

plt.suptitle("CATE Distributions Across 13 Models", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 4b. Qini Curves and AUUC (12 points)

Apply `compute_auuc()` to all 13 models using `Y_te` and `D_te`. Produce **two figures**:

**Figure 1 — All Qini curves.** Plot all 13 curves on one set of axes, colored by model family (meta-learners in blues, causal forest in greens, DML family in reds), with the random baseline as a dashed gray line.

**Figure 2 — Naive vs. tuned within each family.** A 2×3 grid of panels, one per tunable model family (S-Learner, T-Learner, CausalForest, LinearDML, NonParamDML, CausalForestDML). Each panel shows the naive and tuned Qini curves for that family only, labeled with their AUUC values.

Below both figures, print a **ranked AUUC table** (highest to lowest) with columns: Rank, Model, AUUC, AUUC gain vs. naive (for tuned models).

✍️ **Written answer** (4–6 sentences): Across model families, which architectural choice matters most — meta-learner style, causal forest, or DML final stage? Within each family, how consistent is the tuning gain (tuned AUUC minus naive AUUC)? For which family is tuning most impactful, and why? Does the model with the widest CATE spread always rank highest by AUUC — if not, what does this tell you about the relationship between heterogeneity magnitude and ranking quality? Finally, compare `LinearDML` and `NonParamDML`: if they achieve similar AUUC, what does that suggest about the true CATE structure in this dataset?

💻 **Code required**

In [ ]:
# YOUR CODE HERE
# ── Part 4b: Qini Curves and AUUC ───────────────────────────────────────────

# Compute AUUC and Qini curves for all models
auuc_rows = []
qini_results = {}

for name, cate in ALL_MODELS.items():
    auuc, fracs, qini = compute_auuc(Y_te, D_te, np.asarray(cate))
    qini_results[name] = {
        "auuc": auuc,
        "fracs": fracs,
        "qini": qini
    }
    auuc_rows.append({
        "Model": name,
        "AUUC": auuc
    })

# Random baseline is common because qini[-1] is the full-sample effect
first_model = next(iter(qini_results))
fracs_base = qini_results[first_model]["fracs"]
qini_end = qini_results[first_model]["qini"][-1]
random_baseline = np.linspace(0, qini_end, len(fracs_base))


# Color map by model family
model_colors = {
    # Linear benchmark
    "Linear (OLS)": "black",

    # Meta-learners: blues
    "S-Learner naive": "#9ecae1",
    "S-Learner tuned": "#08519c",
    "T-Learner naive": "#6baed6",
    "T-Learner tuned": "#2171b5",

    # Causal forest: greens
    "CausalForest naive": "#a1d99b",
    "CausalForest tuned": "#238b45",

    # DML family: reds
    "LinearDML naive": "#fcae91",
    "LinearDML tuned": "#cb181d",
    "NonParamDML naive": "#fb6a4a",
    "NonParamDML tuned": "#a50f15",
    "CausalForestDML naive": "#fdd0a2",
    "CausalForestDML tuned": "#e34a33",
}

# Different line styles for naive/tuned
def linestyle_for_model(name):
    if "tuned" in name:
        return "-"
    elif "naive" in name:
        return "--"
    else:
        return "-."


# Figure 1 — All Qini curves
fig, ax = plt.subplots(figsize=(11, 7))

for name in ALL_MODELS.keys():
    res = qini_results[name]
    ax.plot(
        res["fracs"],
        res["qini"],
        label=f"{name} (AUUC={res['auuc']:.2f})",
        color=model_colors.get(name, "gray"),
        linestyle=linestyle_for_model(name),
        linewidth=2
    )

ax.plot(
    fracs_base,
    random_baseline,
    color="gray",
    linestyle="--",
    linewidth=2,
    label="Random baseline"
)

ax.set_title("Qini Curves for All 13 CATE Models")
ax.set_xlabel("Fraction of test customers targeted")
ax.set_ylabel("Qini value")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# Figure 2 — Naive vs tuned within each tunable family
families = [
    ("S-Learner", "S-Learner naive", "S-Learner tuned"),
    ("T-Learner", "T-Learner naive", "T-Learner tuned"),
    ("CausalForest", "CausalForest naive", "CausalForest tuned"),
    ("LinearDML", "LinearDML naive", "LinearDML tuned"),
    ("NonParamDML", "NonParamDML naive", "NonParamDML tuned"),
    ("CausalForestDML", "CausalForestDML naive", "CausalForestDML tuned"),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

for ax, (family, naive_name, tuned_name) in zip(axes, families):
    for name in [naive_name, tuned_name]:
        res = qini_results[name]
        ax.plot(
            res["fracs"],
            res["qini"],
            label=f"{name.replace(family + ' ', '')} AUUC={res['auuc']:.2f}",
            color=model_colors.get(name, "gray"),
            linestyle=linestyle_for_model(name),
            linewidth=2
        )

    ax.plot(
        fracs_base,
        random_baseline,
        color="gray",
        linestyle="--",
        linewidth=1.5,
        label="Random"
    )

    ax.set_title(f"{family}: Naive vs Tuned")
    ax.set_xlabel("Fraction targeted")
    ax.set_ylabel("Qini value")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle("Naive vs Tuned Qini Curves by Model Family", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


# Ranked AUUC table with tuning gain
auuc_table = pd.DataFrame(auuc_rows)

naive_lookup = {
    "S-Learner tuned": "S-Learner naive",
    "T-Learner tuned": "T-Learner naive",
    "CausalForest tuned": "CausalForest naive",
    "LinearDML tuned": "LinearDML naive",
    "NonParamDML tuned": "NonParamDML naive",
    "CausalForestDML tuned": "CausalForestDML naive",
}

auuc_dict = dict(zip(auuc_table["Model"], auuc_table["AUUC"]))

def auuc_gain_vs_naive(model_name):
    if model_name in naive_lookup:
        return auuc_dict[model_name] - auuc_dict[naive_lookup[model_name]]
    return np.nan

auuc_table["AUUC gain vs. naive"] = auuc_table["Model"].apply(auuc_gain_vs_naive)

auuc_ranked = (
    auuc_table
    .sort_values("AUUC", ascending=False)
    .reset_index(drop=True)
)

auuc_ranked.insert(0, "Rank", np.arange(1, len(auuc_ranked) + 1))

print("Ranked AUUC table:")
display(
    auuc_ranked.style.format({
        "AUUC": "{:.4f}",
        "AUUC gain vs. naive": "{:.4f}"
    })
)

# Also print plain table for easy copying/debugging
print(auuc_ranked.to_string(index=False))

*Your answer:*Across model families, the most important architectural choice appears to be using flexible nonparametric treatment-effect models, especially causal forests and flexible DML final stages. The best model is the tuned CausalForest with AUUC around 24450.95, while CausalForestDML and tuned NonParamDML also rank near the top. Tuning gains are not equally consistent across families: tuning helps S-Learner, T-Learner, CausalForest, LinearDML, NonParamDML, and CausalForestDML, but the gain is much larger for NonParamDML, T-Learner, and S-Learner than for LinearDML or CausalForestDML. The most impactful tuning gain is for NonParamDML, where AUUC increases by about 10648.53, likely because the flexible final-stage LightGBM model benefits strongly from better hyperparameters when fitting the noisy pseudo-outcome. The model with the widest CATE spread does not always rank highest by AUUC; for example, NonParamDML tuned and T-Learner tuned have very wide CATE distributions, but CausalForest tuned ranks higher, showing that ranking quality matters more than simply producing large heterogeneity. LinearDML and NonParamDML do not achieve similar AUUC here; NonParamDML performs much better, which suggests that the true CATE structure is not fully captured by a simple linear treatment-effect model.

### 4c. Pairwise CATE Agreement (4 points)

Produce a **heatmap** of pairwise Pearson correlations between all 13 CATE vectors on the test set. Annotate each cell with the correlation value. Use a diverging colormap centered at zero.

✍️ **Written answer** (2–3 sentences): Which pairs of models agree most strongly on customer rankings (high correlation)? Which pairs disagree most? Does the naive/tuned pair within the same family always have higher correlation than two different families — and what would it mean if they did not?

💻 **Code required**

In [ ]:
# YOUR CODE HERE
# ── Part 4c: Pairwise CATE Agreement ─────────────────────────────────────────

# Build dataframe of all CATE vectors
cate_matrix = pd.DataFrame({
    name: np.asarray(cate)
    for name, cate in ALL_MODELS.items()
})

# Pearson correlation matrix
cate_corr = cate_matrix.corr(method="pearson")

# Heatmap
fig, ax = plt.subplots(figsize=(13, 11))

im = ax.imshow(
    cate_corr.values,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

# Axis labels
ax.set_xticks(np.arange(len(cate_corr.columns)))
ax.set_yticks(np.arange(len(cate_corr.index)))
ax.set_xticklabels(cate_corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(cate_corr.index, fontsize=9)

# Annotate cells
for i in range(cate_corr.shape[0]):
    for j in range(cate_corr.shape[1]):
        val = cate_corr.iloc[i, j]
        ax.text(
            j, i, f"{val:.2f}",
            ha="center", va="center",
            fontsize=8,
            color="black"
        )

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Pearson correlation")

ax.set_title("Pairwise Pearson Correlations Between CATE Vectors")
plt.tight_layout()
plt.show()


# Print strongest and weakest off-diagonal correlations
pairs = []

models = list(cate_corr.columns)
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        pairs.append({
            "Model 1": models[i],
            "Model 2": models[j],
            "Correlation": cate_corr.iloc[i, j]
        })

corr_pairs = pd.DataFrame(pairs)

print("Highest pairwise correlations:")
display(
    corr_pairs
    .sort_values("Correlation", ascending=False)
    .head(10)
    .style.format({"Correlation": "{:.4f}"})
)

print("Lowest pairwise correlations:")
display(
    corr_pairs
    .sort_values("Correlation", ascending=True)
    .head(10)
    .style.format({"Correlation": "{:.4f}"})
)

*Your answer:*The strongest agreement is among models with similar architecture, especially Linear OLS and LinearDML naive with correlation about 0.993, LinearDML naive and tuned with correlation about 0.972, and CausalForest naive and tuned with correlation about 0.951. The weakest agreement is between linear models and flexible tuned models, such as Linear OLS versus NonParamDML tuned and LinearDML tuned versus NonParamDML tuned, both with correlations around 0.08. The naive/tuned pair within the same family often has high correlation, but this is not always higher than correlations across related families; for example, CausalForest and CausalForestDML also agree strongly. If a within-family naive/tuned pair had lower correlation than two different families, it would mean tuning changed the customer ranking so much that the tuned model behaves more like a different architecture than just a refined version of the naive model.

---
# STAGE 3 — SELECT

---
### 4d. Winner Selection (4 points)

Choose **one model** as your winner from the 13 candidates. Primary criterion: AUUC. Secondary criteria you may also weigh:
- **Calibration:** does the mean CATE match the empirical ATE (difference-in-means on the test set)?
- **Interpretability:** does the model report confidence intervals or feature importances?
- **Tuning gain:** if the tuned version barely improved over naive, is the naive equally suitable?

Set `WINNER_NAME` to one of the keys in `ALL_MODELS`.

✍️ **Required justification** (3–5 sentences): State your winner and address at least two secondary criteria. If the top two models by AUUC are within 1%, explain your tiebreaker.

In [ ]:
# ── SET YOUR WINNER ────────────────────────────────────────────────────────────
WINNER_NAME = 'CausalForest tuned'

assert WINNER_NAME in ALL_MODELS, \
    f"{WINNER_NAME!r} not found. Valid keys: {list(ALL_MODELS)}"

cate_winner = ALL_MODELS[WINNER_NAME]

empirical_ate = Y_te[D_te == 1].mean() - Y_te[D_te == 0].mean()

print(f"Winner : {WINNER_NAME}")
print(f"  Mean : {cate_winner.mean():.4f}")
print(f"  Std  : {cate_winner.std():.4f}")
print(f"  Empirical test ATE: {empirical_ate:.4f}")
print(f"  Calibration gap: {cate_winner.mean() - empirical_ate:.4f}")

*Your justification:*I choose CausalForest tuned as the winner because it has the highest AUUC among all 13 models, about 24450.95. The second-best model is CausalForestDML tuned with AUUC about 22654.28, so the top two are not within 1%, and no tiebreaker is needed. As a calibration check, the tuned CausalForest has a mean CATE of about $0.606, which is reasonably close to the empirical test-set ATE of about $0.530, with a calibration gap of only about $0.076. Tuning also provides a meaningful gain over the naive CausalForest, increasing AUUC by about 2560.61, so the tuned version is preferred over the naive one. Although LinearDML is more interpretable, its AUUC is much lower, so I prioritize targeting performance for the final campaign optimization.

---
# STAGE 4 — OPTIMIZE

All optimization in Part 5 uses only `cate_winner`, `Y_te`, and `D_te`.

---
### Profit Evaluation: Two Methods and Their Assumptions

**The CATE estimates are never ground truth.** Targeting decisions are driven by $\hat{\tau}_i$ (ranking customers by estimated incremental spending), but profit must be evaluated against real outcomes. Both $\hat{\tau}_i$ and $Y_i$ are already in **dollar units**, so no revenue multiplier is needed.

---
**Method 1 — Model-based profit** (assumes $\hat{\tau}_i$ is the true incremental spending):

$$\text{profit}_{\text{model}} = \sum_{i \in \mathcal{T}} (\hat{\tau}_i - c)$$

This is what you optimized against. It measures internal consistency — how good the campaign *looks* according to your model. **It cannot detect miscalibration** because it uses the same estimates that drove the targeting decision.

---
**Method 2 — Empirical profit** (uses observed spending on the test set):

Apply the targeting rule $\mathbf{1}[\hat{\tau}_i > \hat{\tau}^*]$ to **all** test customers — both treated ($D=1$) and control ($D=0$) alike. This partitions the test set into targeted $\mathcal{T}$ and untargeted $\mathcal{U}$. Within $\mathcal{T}$, both arms are present. The control units inside $\mathcal{T}$ serve as the counterfactual for the treated units inside $\mathcal{T}$:

$$\hat{\tau}_{\mathcal{T}} = \underbrace{\bar{Y}_{D=1,\,i\in\mathcal{T}}}_{\approx\,E[Y(1)\mid X\in\mathcal{T}]} - \underbrace{\bar{Y}_{D=0,\,i\in\mathcal{T}}}_{\approx\,E[Y(0)\mid X\in\mathcal{T}]} \quad (\text{in dollars})$$

$$\text{profit}_{\text{empirical}} = |\mathcal{T}|\cdot(\hat{\tau}_{\mathcal{T}} - c)$$

**Required assumptions for Method 2 to be valid:**
1. **Unconfoundedness:** $D \perp Y(0), Y(1) \mid X$ — treatment is quasi-random conditional on covariates. The quasi-random timing of offer arrival conditional on demographics supports this assumption.
2. **Pre-outcome targeting:** $\hat{\tau}_i$ depends only on $X_i$ (model trained on training set, applied to test covariates only).
3. **Overlap within $\mathcal{T}$:** both treated and control units must exist inside the targeted group. Always report $n_{D=1,\mathcal{T}}$ and $n_{D=0,\mathcal{T}}$ as an overlap check — with continuous spending outcomes, small targeted groups can produce very noisy estimates.
4. **SUTVA:** no spillovers between customers.

**Limitations of Method 2 with continuous spending:**
- Spending is right-skewed and potentially heavy-tailed — a few high-spending outliers in the targeted group can dominate $\bar{Y}_{D=1,\mathcal{T}}$ and inflate or deflate the empirical estimate. Consider reporting the median spending gap alongside the mean.
- The variance of the difference-in-means estimator is larger for continuous outcomes than binary ones. With small $|\mathcal{T}|$, the empirical profit estimate may be too noisy to be reliable.
- Spending is right-skewed — a few high-spending outliers in the targeted group can dominate $\bar{Y}_{D=1,\mathcal{T}}$. Consider reporting the median spending gap alongside the mean.

**Limitations of Method 1 with continuous spending:**
- Circular: optimized and evaluated with the same $\hat{\tau}_i$.
- Winner's curse at the cutoff: customers just above $\hat{\tau}^*$ likely have true $\tau_i < \hat{\tau}^*$. With continuous outcomes and a wide CATE distribution, this bias can be substantial near the budget cutoff.
- Model-based profit is particularly unreliable for high-CATE customers: the regression model may overfit to a few high-spending training observations, producing inflated CATE estimates for that subgroup.

**The gap** $\text{profit}_{\text{model}} - \text{profit}_{\text{empirical}}$ measures model optimism. You will compute and interpret this gap throughout Part 5.

Use `profit_model_based()` and `profit_empirical()` for all profit calculations.

## Part 5: Budget-Constrained Targeting (20 points)

### 5a. Unconstrained Optimal Policy (5 points)

Target customer $i$ iff $\hat{\tau}_i > c = \$2.00$. Evaluate under both profit methods.

**Overlap check.** Print $n_{D=1,\mathcal{T}}$ and $n_{D=0,\mathcal{T}}$ for the unconstrained optimal policy. If either is fewer than 50, flag as a reliability concern.

**Report in a single table** (one row per policy: target all / target none / unconstrained optimal):

| Policy | # Targeted | $n_{D=1}$ in $\mathcal{T}$ | $n_{D=0}$ in $\mathcal{T}$ | $\hat{\tau}_{\mathcal{T}}$ (\$) | Profit — model (\$) | Profit — empirical (\$) | Gap (\$) |
|--------|-----------|--------------------------|--------------------------|-------------------------------|--------------------|-----------------------|----------|

Produce a **grouped bar chart** comparing model-based vs. empirical profit for each of the three policies side by side.

✍️ **Written answer** (2–3 sentences): Is the model-based profit higher or lower than the empirical profit for the unconstrained optimal policy, and in which direction does the winner's curse push this gap? Under what condition on $c$ does targeting all customers always dominate targeting none?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# ── Part 5a: Unconstrained Optimal Policy ───────────────────────────────────

# Make sure we are using the selected winner only
print("Using winner model:", WINNER_NAME)

n_test = len(cate_winner)

# Define three policies
target_all_mask = np.ones(n_test, dtype=bool)
target_none_mask = np.zeros(n_test, dtype=bool)
target_opt_mask = cate_winner > COUPON_COST

policies = {
    "Target all": target_all_mask,
    "Target none": target_none_mask,
    "Unconstrained optimal": target_opt_mask
}

# Helper to summarize one policy
def summarize_policy(policy_name, target_mask):
    emp = profit_empirical(Y_te, D_te, target_mask)
    model_profit = profit_model_based(cate_winner, target_mask)
    empirical_profit = emp["profit"]
    gap = model_profit - empirical_profit

    return {
        "Policy": policy_name,
        "# Targeted": int(target_mask.sum()),
        "nD=1 in T": emp["n_treated_in_T"],
        "nD=0 in T": emp["n_control_in_T"],
        "tau_T ($)": emp["tau_empirical"],
        "Profit — model ($)": model_profit,
        "Profit — empirical ($)": empirical_profit,
        "Gap ($)": gap
    }

policy_rows = []

for policy_name, target_mask in policies.items():
    policy_rows.append(summarize_policy(policy_name, target_mask))

policy_table_5a = pd.DataFrame(policy_rows)

display(
    policy_table_5a.style.format({
        "# Targeted": "{:,.0f}",
        "nD=1 in T": "{:,.0f}",
        "nD=0 in T": "{:,.0f}",
        "tau_T ($)": "{:.3f}",
        "Profit — model ($)": "{:,.2f}",
        "Profit — empirical ($)": "{:,.2f}",
        "Gap ($)": "{:,.2f}"
    })
)

# Overlap check for unconstrained optimal policy
opt_emp = profit_empirical(Y_te, D_te, target_opt_mask)

print("\nOverlap check for unconstrained optimal policy:")
print("nD=1 in T:", opt_emp["n_treated_in_T"])
print("nD=0 in T:", opt_emp["n_control_in_T"])

if opt_emp["n_treated_in_T"] < 50 or opt_emp["n_control_in_T"] < 50:
    print("WARNING: Reliability concern — fewer than 50 treated or control units in the targeted group.")
else:
    print("Overlap looks acceptable — both treated and control counts are at least 50.")


# Grouped bar chart: model-based vs empirical profit
plot_df = policy_table_5a[["Policy", "Profit — model ($)", "Profit — empirical ($)"]].copy()

x = np.arange(len(plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    plot_df["Profit — model ($)"],
    width,
    label="Model-based profit"
)

ax.bar(
    x + width / 2,
    plot_df["Profit — empirical ($)"],
    width,
    label="Empirical profit"
)

ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["Policy"], rotation=20, ha="right")
ax.set_ylabel("Profit ($)")
ax.set_title("Model-Based vs Empirical Profit by Policy")
ax.legend()

plt.tight_layout()
plt.show()

*Your answer:*For the unconstrained optimal policy, the model-based profit is lower than the empirical profit: about $15,080 versus about $20,852, giving a negative optimism gap of about -$5,772. In general, the winner's curse tends to push the model-based estimate upward because customers just above the cutoff may have overestimated CATEs, but in this test set the empirical treated-control difference within the targeted group is even larger than the model predicted. Targeting all customers dominates targeting none only when the average treatment effect exceeds the coupon cost, \(ATE > c\); here the empirical ATE is about $0.53, which is below the $2 cost, so targeting all loses money.

### 5b. Profit-Budget Curve (6 points)

Sweep $B$ from $\$0$ to $B_{\max} = N_{\text{test}} \times c$ at 100 evenly spaced points. At each $B$, apply the greedy CATE-ranked policy (sort by $\hat{\tau}$ descending, add customers until the budget is exhausted). Compute **both** model-based and empirical profit at each budget level.

Produce a **two-panel figure** (shared x-axis):
- **Top:** model-based profit (solid blue) and empirical profit (dashed coral) vs. budget. Mark with vertical lines: (i) budget where model-based profit peaks, (ii) budget where empirical profit peaks, (iii) 50% budget.
- **Bottom:** fraction of test set targeted vs. budget. Add a horizontal dashed reference line at the fraction with $\hat{\tau}_i > \$2$.

Print the peak budget levels and the profit at the 50% budget for both methods.

✍️ **Written answer** (2–3 sentences): Do the model-based and empirical profit curves peak at the same budget level — if not, what explains the difference? At the 50% budget level, which profit measure would you report to a business stakeholder, and why?

💻 **Code required**


In [ ]:

# ── Part 5b: Profit-Budget Curve ─────────────────────────────────────────────

print("Using winner model:", WINNER_NAME)

n_test = len(cate_winner)
B_max = n_test * COUPON_COST
B_50 = 0.5 * B_max

# Sort customers by winner CATE descending
order = np.argsort(-cate_winner)

# 100 evenly spaced budget levels
budgets = np.linspace(0, B_max, 100)

curve_rows = []

for B in budgets:
    # Number of customers affordable under budget B
    k = int(np.floor(B / COUPON_COST))
    k = min(k, n_test)

    target_mask = np.zeros(n_test, dtype=bool)
    if k > 0:
        target_mask[order[:k]] = True

    model_profit = profit_model_based(cate_winner, target_mask)
    emp_result = profit_empirical(Y_te, D_te, target_mask)
    empirical_profit = emp_result["profit"]

    curve_rows.append({
        "Budget ($)": B,
        "# Targeted": k,
        "Fraction targeted": k / n_test,
        "Profit — model ($)": model_profit,
        "Profit — empirical ($)": empirical_profit,
        "tau_T ($)": emp_result["tau_empirical"],
        "nD=1 in T": emp_result["n_treated_in_T"],
        "nD=0 in T": emp_result["n_control_in_T"]
    })

budget_curve = pd.DataFrame(curve_rows)

# Identify peak budgets
idx_model_peak = budget_curve["Profit — model ($)"].idxmax()
idx_emp_peak = budget_curve["Profit — empirical ($)"].idxmax()

B_model_peak = budget_curve.loc[idx_model_peak, "Budget ($)"]
B_emp_peak = budget_curve.loc[idx_emp_peak, "Budget ($)"]

model_peak_profit = budget_curve.loc[idx_model_peak, "Profit — model ($)"]
emp_peak_profit = budget_curve.loc[idx_emp_peak, "Profit — empirical ($)"]

# Exact 50% budget evaluation
k_50 = int(np.floor(B_50 / COUPON_COST))
target_50_mask = np.zeros(n_test, dtype=bool)
target_50_mask[order[:k_50]] = True

model_profit_50 = profit_model_based(cate_winner, target_50_mask)
emp_50 = profit_empirical(Y_te, D_te, target_50_mask)
emp_profit_50 = emp_50["profit"]

frac_break_even = np.mean(cate_winner > BREAK_EVEN)

print("Peak budget levels:")
print(f"Model-based peak budget: ${B_model_peak:,.2f}")
print(f"Model-based peak profit: ${model_peak_profit:,.2f}")
print(f"Empirical peak budget:   ${B_emp_peak:,.2f}")
print(f"Empirical peak profit:   ${emp_peak_profit:,.2f}")

print("\n50% budget results:")
print(f"50% budget: ${B_50:,.2f}")
print(f"# Targeted at 50% budget: {k_50:,}")
print(f"Fraction targeted at 50% budget: {k_50 / n_test:.2%}")
print(f"Model-based profit at 50% budget: ${model_profit_50:,.2f}")
print(f"Empirical profit at 50% budget:   ${emp_profit_50:,.2f}")
print(f"Empirical tau_T at 50% budget: ${emp_50['tau_empirical']:.3f}")
print(f"nD=1 in T: {emp_50['n_treated_in_T']:,}")
print(f"nD=0 in T: {emp_50['n_control_in_T']:,}")

# Display a small summary table
budget_summary_5b = pd.DataFrame({
    "Quantity": [
        "Model-based peak budget",
        "Model-based peak profit",
        "Empirical peak budget",
        "Empirical peak profit",
        "50% budget",
        "Model-based profit at 50% budget",
        "Empirical profit at 50% budget"
    ],
    "Value": [
        B_model_peak,
        model_peak_profit,
        B_emp_peak,
        emp_peak_profit,
        B_50,
        model_profit_50,
        emp_profit_50
    ]
})

display(
    budget_summary_5b.style.format({
        "Value": "${:,.2f}"
    })
)

# Two-panel figure
fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]}
)

# Top panel: profit curves
axes[0].plot(
    budget_curve["Budget ($)"],
    budget_curve["Profit — model ($)"],
    color="blue",
    linestyle="-",
    linewidth=2,
    label="Model-based profit"
)

axes[0].plot(
    budget_curve["Budget ($)"],
    budget_curve["Profit — empirical ($)"],
    color="coral",
    linestyle="--",
    linewidth=2,
    label="Empirical profit"
)

axes[0].axvline(
    B_model_peak,
    color="blue",
    linestyle=":",
    linewidth=2,
    label="Model-based peak"
)

axes[0].axvline(
    B_emp_peak,
    color="coral",
    linestyle=":",
    linewidth=2,
    label="Empirical peak"
)

axes[0].axvline(
    B_50,
    color="black",
    linestyle="-.",
    linewidth=2,
    label="50% budget"
)

axes[0].axhline(0, color="gray", linewidth=1)

axes[0].set_ylabel("Profit ($)")
axes[0].set_title("Profit-Budget Curve Using Winner CATE Ranking")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bottom panel: fraction targeted
axes[1].plot(
    budget_curve["Budget ($)"],
    budget_curve["Fraction targeted"],
    color="black",
    linewidth=2,
    label="Fraction targeted"
)

axes[1].axhline(
    frac_break_even,
    color="gray",
    linestyle="--",
    linewidth=2,
    label=f"Fraction with CATE > $2 = {frac_break_even:.1%}"
)

axes[1].axvline(
    B_50,
    color="black",
    linestyle="-.",
    linewidth=2
)

axes[1].set_xlabel("Budget ($)")
axes[1].set_ylabel("Fraction targeted")
axes[1].set_title("Fraction of Test Set Targeted vs. Budget")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

*Your answer:*The model-based and empirical profit curves peak at similar but not identical budget levels: the model-based peak occurs around $10,303, while the empirical peak occurs around $9,697. This difference is expected because the model-based curve uses the estimated CATEs that also drive the ranking, while the empirical curve uses realized treated-control spending differences and is affected by sampling noise and possible CATE miscalibration. At the 50% budget level, I would report the empirical profit to a business stakeholder because it is based on observed test-set outcomes and is less circular than evaluating the campaign with the same model scores used to choose the customers.

### 5c. Budget Cutoff and Fairness-Constrained LP (9 points)

Set $B = 0.5 \times B_{\max}$.

#### Part i — Greedy budget cutoff (3 points)

The optimal policy under a hard budget with equal coupon costs is simply to target the top-$k$ customers by $\hat{\tau}_i$ where $k = \lfloor B/c \rfloor$. The marginal CATE at position $k$ is the budget cutoff $\hat{\tau}^*$.

Report: $\hat{\tau}^*$, number targeted, total cost, and both profit methods with overlap check.

#### Part ii — Fairness-constrained LP (6 points)

Add a demographic parity constraint: the targeting rate for female customers must be within $\pm 5$ pp of the rate for male customers. Each customer receives a coupon with probability $p_i \in [0,1]$. Solve:

$$\max_{p} \sum_i p_i (\hat{\tau}_i - c)$$

subject to:

$$\sum_i p_i c \leq B, \qquad \left|\frac{\sum_{i \in S_F} p_i}{|S_F|} - \frac{\sum_{i \in S_M} p_i}{|S_M|}\right| \leq 0.05, \qquad 0 \leq p_i \leq 1$$

using `scipy.optimize.linprog`. Binarize by setting $p_i \geq 0.5$ to 1.

Report in a **side-by-side table** (unconstrained | fairness LP):
- Number targeted, targeting rates by gender (Female / Male / Other)
- $n_{D=1}$ and $n_{D=0}$ within $\mathcal{T}$ (overlap check)
- Model-based profit (\\$), empirical profit (\\$), gap (\\$)
- **Cost of fairness:** reduction in empirical profit

Produce a **two-panel figure:**
- **Left:** overlapping histograms of $\hat{\tau}_i$ by gender group, with a vertical dashed line at $\hat{\tau}^*$
- **Right:** bar chart comparing empirical profit under unconstrained vs. fairness LP

✍️ **Written answer** (3–4 sentences): Explain why the greedy cutoff in Part i is the optimal solution to the unconstrained LP (bang-bang structure). Which gender group is targeted more heavily by the unconstrained policy, and what does the CATE histogram tell you about why? Is the cost of fairness economically meaningful given the empirical profit estimates?

💻 **Code required**


In [ ]:
# YOUR CODE HERE
# ── Part 5c: Budget Cutoff and Fairness-Constrained LP ───────────────────────
from scipy.optimize import linprog

print("Using winner model:", WINNER_NAME)

# 50% budget
B = 0.5 * B_max
n = len(cate_winner)
k = int(np.floor(B / COUPON_COST))

# ── Part i: Greedy budget cutoff ─────────────────────────────────────────────
order = np.argsort(-cate_winner)
sorted_cate = cate_winner[order]

tau_star = sorted_cate[k - 1]

mask_greedy = np.zeros(n, dtype=bool)
mask_greedy[order[:k]] = True

cost_greedy = mask_greedy.sum() * COUPON_COST
mb_greedy = profit_model_based(cate_winner, mask_greedy)
emp_greedy = profit_empirical(Y_te, D_te, mask_greedy)

print("Part i — Greedy budget cutoff")
print(f"Budget B: ${B:,.2f}")
print(f"k = floor(B / c): {k:,}")
print(f"Budget cutoff tau*: ${tau_star:.4f}")
print(f"Number targeted: {mask_greedy.sum():,}")
print(f"Total cost: ${cost_greedy:,.2f}")
print(f"Model-based profit: ${mb_greedy:,.2f}")
print(f"Empirical profit: ${emp_greedy['profit']:,.2f}")
print(f"Empirical tau_T: ${emp_greedy['tau_empirical']:.4f}")
print(f"Overlap check: nD=1 in T = {emp_greedy['n_treated_in_T']:,}, "
      f"nD=0 in T = {emp_greedy['n_control_in_T']:,}")

if emp_greedy["n_treated_in_T"] < 50 or emp_greedy["n_control_in_T"] < 50:
    print("WARNING: Reliability concern — fewer than 50 treated or control units.")
else:
    print("Overlap looks acceptable — both treated and control counts are at least 50.")


# ── Gender masks ─────────────────────────────────────────────────────────────
gender_f_col = FEATURE_COLS.index("gender_F")
gender_m_col = FEATURE_COLS.index("gender_M")
gender_o_col = FEATURE_COLS.index("gender_O")

is_F = Xf_te[:, gender_f_col].astype(bool)
is_M = Xf_te[:, gender_m_col].astype(bool)
is_O = Xf_te[:, gender_o_col].astype(bool)

nF = int(is_F.sum())
nM = int(is_M.sum())
nO = int(is_O.sum())

print("\nGender group sizes in test set:")
print(f"Female: {nF:,}")
print(f"Male:   {nM:,}")
print(f"Other:  {nO:,}")


# ── Part ii: Fairness-constrained LP ─────────────────────────────────────────
# Objective: maximize sum p_i * (tau_i - c)
# linprog minimizes, so minimize negative profit
c_obj = -(cate_winner - COUPON_COST)

# Budget constraint: sum p_i * c <= B
budget_row = COUPON_COST * np.ones(n)

# Demographic parity: |rate_F - rate_M| <= 0.05
parity_row = np.zeros(n)
parity_row[is_F] = 1.0 / nF
parity_row[is_M] = -1.0 / nM

A_ub = np.vstack([
    budget_row,
    parity_row,
    -parity_row
])

b_ub = np.array([
    B,
    0.05,
    0.05
])

bounds = [(0, 1)] * n

res = linprog(
    c=c_obj,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs"
)

print("\nFairness LP status:")
print(res.message)

if not res.success:
    raise RuntimeError("Fairness LP did not solve successfully.")

p_fair = res.x

print(f"LP objective value, model profit before binarization: ${-res.fun:,.2f}")
print(f"Mean p_F: {p_fair[is_F].mean():.4f}")
print(f"Mean p_M: {p_fair[is_M].mean():.4f}")
print(f"Difference p_F - p_M: {p_fair[is_F].mean() - p_fair[is_M].mean():.4f}")

# Binarize by p_i >= 0.5
mask_fair = p_fair >= 0.5

# Basic budget check after binarization
cost_fair = mask_fair.sum() * COUPON_COST
if cost_fair > B + 1e-8:
    print("WARNING: Binarized LP exceeds budget. Trimming lowest-CATE selected customers to fit budget.")
    selected_idx = np.where(mask_fair)[0]
    selected_sorted = selected_idx[np.argsort(cate_winner[selected_idx])]  # low CATE first
    excess = int(np.ceil((cost_fair - B) / COUPON_COST))
    mask_fair[selected_sorted[:excess]] = False
    cost_fair = mask_fair.sum() * COUPON_COST

mb_fair = profit_model_based(cate_winner, mask_fair)
emp_fair = profit_empirical(Y_te, D_te, mask_fair)

print("\nFairness LP after binarization:")
print(f"Number targeted: {mask_fair.sum():,}")
print(f"Total cost: ${cost_fair:,.2f}")
print(f"Model-based profit: ${mb_fair:,.2f}")
print(f"Empirical profit: ${emp_fair['profit']:,.2f}")
print(f"Empirical tau_T: ${emp_fair['tau_empirical']:.4f}")
print(f"Overlap check: nD=1 in T = {emp_fair['n_treated_in_T']:,}, "
      f"nD=0 in T = {emp_fair['n_control_in_T']:,}")

if emp_fair["n_treated_in_T"] < 50 or emp_fair["n_control_in_T"] < 50:
    print("WARNING: Reliability concern — fewer than 50 treated or control units.")
else:
    print("Overlap looks acceptable — both treated and control counts are at least 50.")


# ── Helper for side-by-side policy summary ───────────────────────────────────
def gender_rate(mask, group_mask):
    if group_mask.sum() == 0:
        return np.nan
    return float(mask[group_mask].mean())

def policy_summary(label, mask):
    emp = profit_empirical(Y_te, D_te, mask)
    mb = profit_model_based(cate_winner, mask)
    return {
        "Policy": label,
        "Number targeted": int(mask.sum()),
        "Total cost ($)": float(mask.sum() * COUPON_COST),
        "Female targeting rate": gender_rate(mask, is_F),
        "Male targeting rate": gender_rate(mask, is_M),
        "Other targeting rate": gender_rate(mask, is_O),
        "F - M targeting rate": gender_rate(mask, is_F) - gender_rate(mask, is_M),
        "nD=1 in T": emp["n_treated_in_T"],
        "nD=0 in T": emp["n_control_in_T"],
        "tau_T ($)": emp["tau_empirical"],
        "Model-based profit ($)": mb,
        "Empirical profit ($)": emp["profit"],
        "Gap ($)": mb - emp["profit"]
    }

summary_5c = pd.DataFrame([
    policy_summary("Unconstrained greedy", mask_greedy),
    policy_summary("Fairness LP", mask_fair)
])

cost_of_fairness = (
    summary_5c.loc[summary_5c["Policy"] == "Unconstrained greedy", "Empirical profit ($)"].iloc[0]
    - summary_5c.loc[summary_5c["Policy"] == "Fairness LP", "Empirical profit ($)"].iloc[0]
)

summary_5c["Cost of fairness ($)"] = [
    np.nan,
    cost_of_fairness
]

print("\nSide-by-side policy comparison:")
display(
    summary_5c.style.format({
        "Number targeted": "{:,.0f}",
        "Total cost ($)": "${:,.2f}",
        "Female targeting rate": "{:.2%}",
        "Male targeting rate": "{:.2%}",
        "Other targeting rate": "{:.2%}",
        "F - M targeting rate": "{:.2%}",
        "nD=1 in T": "{:,.0f}",
        "nD=0 in T": "{:,.0f}",
        "tau_T ($)": "${:.3f}",
        "Model-based profit ($)": "${:,.2f}",
        "Empirical profit ($)": "${:,.2f}",
        "Gap ($)": "${:,.2f}",
        "Cost of fairness ($)": "${:,.2f}"
    })
)

print(f"\nCost of fairness, empirical profit reduction: ${cost_of_fairness:,.2f}")


# ── Two-panel figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping histograms of CATE by gender group
axes[0].hist(cate_winner[is_F], bins=50, alpha=0.5, label="Female")
axes[0].hist(cate_winner[is_M], bins=50, alpha=0.5, label="Male")
axes[0].hist(cate_winner[is_O], bins=50, alpha=0.5, label="Other")

axes[0].axvline(
    tau_star,
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"tau* = ${tau_star:.2f}"
)

axes[0].set_title("Winner CATE Distribution by Gender")
axes[0].set_xlabel("Predicted CATE ($)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Right: empirical profit comparison
profit_plot = summary_5c[["Policy", "Empirical profit ($)"]]

axes[1].bar(
    profit_plot["Policy"],
    profit_plot["Empirical profit ($)"]
)

axes[1].axhline(0, color="gray", linewidth=1)
axes[1].set_title("Empirical Profit: Unconstrained vs Fairness LP")
axes[1].set_ylabel("Empirical profit ($)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

*Your answer:*The greedy cutoff is optimal for the unconstrained hard-budget problem with equal coupon costs because every coupon has the same cost, so the best use of a fixed number of coupons is to assign them to the customers with the highest predicted CATE. Under the unconstrained greedy policy, the Other group and Female group are targeted more heavily than Male customers: about 71.0% for Other, 63.8% for Female, and 52.4% for Male, and the CATE histogram suggests that Female and Other customers have more mass above the cutoff. The fairness LP reduces the Female-Male targeting-rate gap from about 11.5 percentage points to exactly 5.0 percentage points. In this run, the “cost of fairness” is not economically negative; the fairness LP actually increases empirical profit by about $12,401, mainly because it does not spend the full budget and avoids many low- or negative-margin customers that the 50% greedy budget rule includes.

---
## Part 6: Reflection (required)

Answer each question in 3–5 sentences.

### 6a. Tuning Gains and Their Source

Looking at your AUUC table from Part 4b, for which model family was the tuning gain largest? For S and T learners, lower RMSE on spending prediction does not guarantee better CATE ranking — explain why, and describe what property of the spending DGP would make outcome-model tuning reliably improve CATE quality. Does switching from binary conversion to continuous spending make tuning more or less impactful, and why?

✍️ **Written answer**

*Your answer:*The largest tuning gain in my AUUC table was for NonParamDML, where AUUC increased by about 10648.53 after tuning. For S- and T-learners, lower RMSE on spending prediction does not guarantee better CATE ranking because CATE is based on the difference between predicted spending under treatment and control, not the accuracy of either spending prediction alone. Outcome-model tuning would reliably improve CATE quality only if the spending DGP contains treatment-covariate interactions that the tuned model captures better, since those interactions determine heterogeneous treatment effects. Switching from binary conversion to continuous spending likely makes tuning more impactful because spending is noisier and more heavy-tailed, so model flexibility and regularization matter more when predicting dollar outcomes.

### 6b. LinearDML vs. NonParamDML

If `LinearDML` and `NonParamDML` achieved similar AUUC in your results, what does that imply about the shape of the true CATE in this dataset — is the heterogeneity approximately linear in age, income, and gender? Conversely, if `NonParamDML` significantly outperformed `LinearDML`, what nonlinear interaction does that suggest exists? If you were deploying in a business setting and the two models had equal AUUC, which would you choose and why?

✍️ **Written answer**

*Your answer:*In my results, NonParamDML significantly outperformed LinearDML: tuned NonParamDML achieved AUUC around 22524.76, while tuned LinearDML achieved only about 2670.42. This suggests that the true CATE structure is not well captured by a simple linear function of age, income, and gender. The improvement from NonParamDML suggests there may be nonlinear interactions among demographics, such as income effects differing by gender or treatment effects changing nonlinearly across age groups. If the two models had equal AUUC in a business setting, I would choose LinearDML because it is simpler, more interpretable, and provides coefficient inference, making it easier to explain targeting decisions to stakeholders.

### 6c. The Feedback Loop Problem

Suppose you deploy the greedy 50% budget policy for three months and retrain the winning CATE model on the new data. Targeted customers all received a coupon — their observed spending reflects both organic behavior and the offer effect. Untargeted customers received no coupon. How does this selective observation bias the new training data, particularly for the high-income or high-CATE subgroup that was disproportionately targeted? Propose one practical mitigation that does not require redesigning the campaign.

✍️ **Written answer**

*Your answer:*Retraining after deploying the greedy 50% budget policy creates selective observation bias because high-CATE customers are much more likely to receive coupons, while lower-ranked customers are less likely to be treated. For high-income or high-CATE subgroups that were disproportionately targeted, their observed spending would mix organic demand with the coupon effect, making it hard to distinguish whether they spent more because they are naturally valuable or because the offer caused additional spending. This can make the model overconfident about repeatedly targeting the same subgroup and reduce learning about customers who were not selected. A practical mitigation is to keep a small randomized holdout group inside each major segment, where some customers are intentionally not treated or are randomly treated, so the company continues to collect counterfactual information without redesigning the whole campaign.

---
## Submission Checklist

- [ ] All code cells run top-to-bottom without errors (Kernel → Restart & Run All)
- [ ] All figures visible in output; all written answers filled in
- [ ] Setup, Data Loading, and Business Parameters cells not modified
- [ ] Part 1: `cate_linear`, `cate_s_naive`, `cate_s_tuned`, `cate_t_naive`, `cate_t_tuned` stored; `LGBMRegressor` used throughout (not classifier)
- [ ] Part 2: `cate_cf_naive`, `cate_cf_tuned` stored; `r_loss_cv()` implemented; best hyperparameters printed
- [ ] Part 3: `model_y` is `LGBMRegressor` tuned by MSE; `model_t` is `LGBMClassifier` tuned by log-loss; `Y_tilde_tr` and `D_tilde_tr` stored; nuisance summary table reported; all six DML CATE arrays stored
- [ ] Part 4: `ALL_MODELS` dict has exactly 13 entries; `WINNER_NAME` set to a valid key; written answer justifies choice
- [ ] Part 5: all optimization uses `cate_winner`; both `profit_model_based()` and `profit_empirical()` used; overlap check reported in 5a and 5c; fairness LP solved with `linprog`; cost of fairness reported
- [ ] Part 6: all three reflection questions answered

---
## Grading Rubric

| Part | Points | Key Criteria |
|------|--------|--------------|
| **1. Linear Benchmark and Meta-Learners** | **25** | |
| 1a. Linear interaction model | 5 | OLS on continuous spending; correct interaction design matrix; significant terms identified; `cate_linear` in dollars |
| 1b. S-Learner naive + tuned | 10 | `LGBMRegressor`; `GridSearchCV` on RMSE; CATE histogram; written answer on why RMSE ≠ CATE ranking |
| 1c. T-Learner naive + tuned | 10 | Independent per-arm tuning on RMSE; CATE histogram; written answer on arm complexity |
| **2. Causal Forest** | **20** | |
| 2a. CausalForest naive + tuned | 20 | `r_loss_cv()` implemented and used for tuning; correct positional `fit(X,T,y)` API; CATE histogram; written answer on R-loss vs. RMSE |
| **3. DML Family** | **30** | |
| 3a. Shared nuisance tuning | 8 | `model_y` = `LGBMRegressor` tuned by MSE; `model_t` = `LGBMClassifier` tuned by log-loss; nuisance summary table; `Y_tilde_tr` and `D_tilde_tr` saved via 5-fold cross-fitting |
| 3b. LinearDML naive + tuned | 6 | Correct `econml` API; `.coef__inference().pvalue()` called correctly; coefficient table; CATE histogram |
| 3c. NonParamDML naive + tuned | 8 | Correct pseudo-outcome and `sample_weight`; `GridSearchCV.fit(..., sample_weight=...)` syntax; CATE histogram |
| 3d. CausalForestDML naive + tuned | 8 | `n_estimators` divisible by 4; `.tune()` then `.fit()`; CATE histogram; written answer on two tuning criteria |
| **4. Evaluation and Selection** | **25** | |
| 4a. CATE distribution summary | 5 | Table in dollar units; break-even at \$2; 13-panel histogram |
| 4b. Qini curves and AUUC | 12 | All 13 models; two Qini figures (all-in-one + within-family); ranked AUUC table with tuning gain column; written answer |
| 4c. Pairwise correlation heatmap | 4 | Correct Pearson correlations; diverging colormap; written answer |
| 4d. Winner selection | 4 | Valid `WINNER_NAME`; justification addresses AUUC and at least two secondary criteria |
| **5. Budget-Constrained Targeting** | **20** | |
| 5a. Unconstrained policy | 5 | Three-policy table with both profit methods and overlap counts; grouped bar chart; written answer on winner's curse direction |
| 5b. Profit-budget curve | 6 | Both profit curves on same axes; two-panel figure with peak markers; written answer on stakeholder reporting |
| 5c. Budget cutoff + fairness LP | 9 | Cutoff $\hat{\tau}^*$ correctly found; `linprog` for fairness LP; side-by-side table with cost of fairness; CATE-by-gender histogram; written answer on bang-bang structure |
| **Total** | **120** | |

*Part 6 (Reflection) is graded holistically. Shallow or missing answers reduce the total by up to 5 points.*


In [ ]:
# ── Final validation before submission ───────────────────────────────────────

required_cates = {
    "cate_linear": cate_linear,
    "cate_s_naive": cate_s_naive,
    "cate_s_tuned": cate_s_tuned,
    "cate_t_naive": cate_t_naive,
    "cate_t_tuned": cate_t_tuned,
    "cate_cf_naive": cate_cf_naive,
    "cate_cf_tuned": cate_cf_tuned,
    "cate_ldml_naive": cate_ldml_naive,
    "cate_ldml_tuned": cate_ldml_tuned,
    "cate_rl_naive": cate_rl_naive,
    "cate_rl_tuned": cate_rl_tuned,
    "cate_cfdml_naive": cate_cfdml_naive,
    "cate_cfdml_tuned": cate_cfdml_tuned,
}

for name, arr in required_cates.items():
    assert isinstance(arr, np.ndarray), f"{name} is not a numpy array"
    assert arr.shape == (len(Xs_te),), f"{name} has wrong shape: {arr.shape}"

assert len(ALL_MODELS) == 13, f"ALL_MODELS should have 13 entries, got {len(ALL_MODELS)}"
assert WINNER_NAME in ALL_MODELS, f"Invalid WINNER_NAME: {WINNER_NAME}"
assert np.allclose(cate_winner, ALL_MODELS[WINNER_NAME])

assert isinstance(Y_tilde_tr, np.ndarray)
assert isinstance(D_tilde_tr, np.ndarray)
assert Y_tilde_tr.shape == Y_tr.shape
assert D_tilde_tr.shape == D_tr.shape

print("All final validation checks passed.")
print("Winner:", WINNER_NAME)
print("Number of models in ALL_MODELS:", len(ALL_MODELS))
print("Test CATE length:", len(cate_winner))